# Inner-speech EEG: ML, scratch DL and pretrained transfer under channel reduction

A standalone Kaggle experiment using **the same original INNER SPEECH trials, fixed spatial channel sets and outer folds** for all model families. Default: subject 01, all available sessions pooled within that subject, 128/64/32/16 channels, 10 outer folds and seed 42. No previous split directory, feature exports, notebook import or local project installation is required.

| Branch | Representation | What is trained | Evaluation |
|---|---|---|---|
| ML: PAPER_ORDER | Current wavelet/AR/power features, MRMR K=590 | SVM after full-subject supervised selection | **Intentionally leaky diagnostic** |
| ML: NESTED_SAFE | Same candidate features and fixed K | Training-only selector, scaler and SVM | Untouched outer test trials |
| DL | EEG time series | Compact 1D CNN from random initialization | Inner epoch selection, then outer-train refit |
| PRETRAINED | EEG time series processed by official CBraMod weights | Default: logistic regression on frozen embeddings; optional: encoder fine-tuning | Training-only classifier/scaler or inner epoch selection |

PAPER_ORDER deliberately uses held-out target labels in selection; it is not a valid generalization estimate. It diagnoses the effect of ordering and does not establish how the published authors implemented their code. DL/pretrained results should primarily be compared against **ML NESTED_SAFE**.

This is an adapted comparison study, not an exact reproduction of Tasin et al.'s DL experiments. The scratch CNN follows their two-convolution idea; widths/head are our declared choices. CBraMod is an external EEG foundation model, not an ImageNet image classifier. Current ML features remain 120/channel, not the paper's unavailable exact 191-feature definitions.

Selecting subsets of provider-processed 128-channel data simulates available-channel restrictions in software. It does not recreate a low-density recording or rerun its reference/ICA preprocessing.


## 1. Runtime setup

Enable **GPU** and **Internet** in Kaggle before the first pretrained run. The official checkpoint is about 20 MB. For offline use, upload `pretrained_weights.pth` from the linked official Hub repository as a Kaggle dataset and set `pretrained.checkpoint_path`; the same pinned file hash is checked. A missing checkpoint fails before expensive ML training; random weights are never presented as pretrained.

Kaggle usually already provides PyTorch and the scientific stack. If an import is missing, set `INSTALL_MISSING=True` below. This installs missing packages only; it does not upgrade an existing Torch/CUDA environment. Restart the kernel if pip asks you to.

References: [Tasin et al.](https://doi.org/10.1109/ACCESS.2025.3644494), [official CBraMod source](https://github.com/wjq-learning/CBraMod), [official checkpoint linked by the authors](https://huggingface.co/weighting666/CBraMod), [feature-selection leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage-during-pre-processing).


In [ ]:
INSTALL_MISSING = False
import importlib.util
import subprocess
import sys
required = {"numpy":"numpy", "pandas":"pandas", "scipy":"scipy", "sklearn":"scikit-learn",
            "matplotlib":"matplotlib", "mne":"mne", "pywt":"PyWavelets", "torch":"torch"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing and INSTALL_MISSING:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
elif missing:
    raise ImportError(f"Missing packages: {missing}. Set INSTALL_MISSING=True and rerun this cell.")


In [ ]:
"""Subject-specific paper-order diagnostic and training-only evaluation."""
from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction
from types import SimpleNamespace
from time import perf_counter
from zipfile import ZipFile, ZIP_DEFLATED
from itertools import combinations
from importlib.metadata import version, PackageNotFoundError
import hashlib
import importlib.util
import json
import platform
import re
import warnings
import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.signal import welch
from scipy.integrate import trapezoid
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, recall_score
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import pywt
BANDS = ((1.,4.), (4.,8.), (8.,13.), (13.,30.), (30.,45.))
PAPER_LABEL = 'PAPER_ORDER — feature selection before CV; potential leakage'
SAFE_LABEL = 'NESTED_SAFE'
LABELS = [0, 1, 2, 3]
REFERENCES = {
    ('mrmr', 10, 'svm'): (.6653, .6709),
    ('anova', 10, 'lgbm'): (.4594, .4630),
    ('mrmr', 590, 'svm'): (.7631, None),
    ('mrmr', 590, 'lda'): (.7191, None),
    ('mrmr', 590, 'etc'): (.7593, None),
    ('mrmr', 590, 'rf'): (.7342, None),
    ('mrmr', 590, 'lr'): (.7789, None),
    ('mrmr', 590, 'lgbm'): (.7533, None),
    ('mrmr', 590, 'lr5'): (.8113, .8112),
}
_sklearn_mutual_info_classif = mutual_info_classif
"""Fixed spatial channel sets and paired evaluation-boundary comparisons."""
from copy import deepcopy
CHANNEL_EXTENSION_VERSION='channel-reduction-dual-1.0'
CHANNEL_MODEL_NAMES={'svm','lda','etc','rf','lr','lgbm','lr5'}
CHANNEL_BOUNDARY_COLORS={'PAPER_ORDER':'#d95f02','NESTED_SAFE':'#1b9e77'}
ENGINE_ID='0cdf49430e3ad49da85a93ebf9a092e8153894791bd0dc1cc66cb6452d952cdd'
SOURCE_INSPECTION={'current_source': 'Thinking_Out_Loud_Within_Session_ML_Ensemble / audited current source', 'current_source_sha256': 'c168a2f9351deb2b19d6060d219a4233aacdfb688a933d83ecf4a5f941994511', 'current_wavelet_sha256': 'e7d884d0c48ea58ba5102a0bfe27497b84a8eae01e4c7b064376d1bbdc814b82', 'paper_inspired_source': 'C:\\Users\\User\\Programming\\Learning\\python\\braincodecamp\\braincodecamp2026\\paper-reproduce.ipynb', 'paper_inspired_source_sha256': '0622a815f14cbd30f173a45f07a4859174a2eba9807eddbc827d3adb96add733', 'paper_exact_191_available': False, 'paper_pdf': 'C:\\Users\\User\\Programming\\Learning\\python\\braincodecamp\\braincodecamp2026\\presentations\\papers\\Ensemble_Machine_Learning_Model_for_Inner_Speech_Recognition_A_Subject-Specific_Investigation.pdf', 'doi': '10.1109/ACCESS.2025.3644494', 'current_default_reason': 'The saved compare-wavelet-ml-mrmr-anova run uses combined features.', 'loader_adaptation': 'Validated derivatives loader; retain pooled EEG and session IDs; crop by FIF times.', 'no_existing_files_modified': True, 'channel_configuration': 'Canonical BioSemi128 farthest-point spatial rule, matching inspected channel_subsets; nested sets frozen without EEG/labels.', 'previous_channel_notebook': 'No prior notebook is imported at runtime; code is embedded.', 'previous_audited_engine_id': 'f1c2025bfe88cbb552c1ec9e342a8d14d0ce5970f26e5d035e930ca65902f27b'}
from IPython.display import display, FileLink
print("Python:",platform.python_version())
print("Engine:",ENGINE_ID[:16])


## 2. Configuration — edit here, then Run All

- `run_ml`, `run_dl`, `run_pretrained` switch model families on/off.
- `subjects=[1]` initially; `subjects=None` discovers all available subjects.
- `evaluation_scope="pooled_subject"` matches the previous channel notebook: sessions mix across folds within each subject. Use `"within_session"` to train/test separately inside each session. In both modes, subjects have separate models. Neither mode is LOSO or leave-one-session-out.
- `channel_counts=[128,64,32,16]` uses the same geometry rule as our previous notebook; keep 128 as the reference. For a short end-to-end check, use `[128,16]`, `outer_folds=5`, `dl.max_epochs=10`. This changes the protocol, so label it a pilot.
- `classifiers=["svm"]`, `selectors=["mrmr"]`, `k_features=590` control ML. Add `anova`, `lr`, `lda` or `lr5` as needed. Optional `tune_svm=True` uses training-only inner folds for the safe branch. In PAPER_ORDER, its prior full-unit selection remains leaky.
- `pretrained.modes=["frozen_probe"]` is the default transfer baseline. Add `"finetune"` or replace the list with `["finetune"]` to train the encoder and classifier. This costs substantially more, especially with 128 channels.
- CNN and fine-tuning choose **epoch duration only**, using the first predeclared stratified inner split. They then reset and train on all outer-training trials for that duration. They do not search architectures or hyperparameters.
- The full 2.5-second interval is preserved. Neural inputs become 500 samples at 200 Hz; CBraMod adds 100 fixed zero samples to form three 200-sample patches. The CNN receives the original 500 samples.
- No randomized-label runs are enabled in this first three-family notebook. Five controls from the old ML-only notebook are not automatically included. A chance line is only a reference, not a significance test.

Inspect data, workload and channel plots before running the three training sections. Avoid choosing settings by repeatedly inspecting outer-test scores.


In [ ]:
# Edit this single dictionary, then rerun from initialization onward.
CONFIG = {'data_root': None,
 'output_root': '/kaggle/working/inner_speech_three_pipeline',
 'subjects': [1],
 'sessions': None,
 'channel_counts': [128, 64, 32, 16],
 'channel_method': 'spatial',
 'manual_channel_sets': {},
 'boundaries': ['PAPER_ORDER', 'NESTED_SAFE'],
 'selectors': ['mrmr'],
 'classifiers': ['svm'],
 'k_features': 590,
 'seed': 42,
 'outer_folds': 10,
 'inner_folds': 3,
 'action_window_mne': (0.5, 3.0),
 'feature_mode': 'current',
 'current_features': {'feature_set': 'combined',
                      'n_windows': 8,
                      'wavelet': 'db2',
                      'wavelet_level': 3,
                      'ar_order': 4,
                      'feature_sfreq': 128.0,
                      'wavelet_bandpass': [2.0, 45.0]},
 'paper_inspired_features': {'wavelet': 'db4', 'wavelet_level': 5, 'spectral_high': 100.0},
 'vmd_mode': 'off',
 'vmd_manual_channels': {},
 'vmd_robust_z': 4.0,
 'vmd_parameters': {'alpha': 2000.0, 'tau': 0.0, 'dc': 0, 'init': 1, 'tol': 1e-07},
 'mrmr_relevance': 'f_stat',
 'mrmr_corr_floor': 1e-06,
 'tune_svm': False,
 'svm_inner_grid': {'C': [0.1, 1.0, 10.0], 'kernel': ['rbf'], 'gamma': ['scale']},
 'models': {'svm': {'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale', 'cache_size': 512},
            'lda': {'solver': 'lsqr', 'shrinkage': 'auto'},
            'etc': {'n_estimators': 300, 'max_features': 'sqrt'},
            'rf': {'n_estimators': 300, 'max_features': 'sqrt'},
            'lr': {'C': 1.0, 'solver': 'lbfgs', 'max_iter': 5000},
            'lgbm': {'n_estimators': 300,
                     'learning_rate': 0.05,
                     'num_leaves': 15,
                     'deterministic': True,
                     'force_col_wise': True}},
 'lr5_members': [{'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'},
                 {'C': 1.0, 'penalty': 'l2', 'solver': 'lbfgs'},
                 {'C': 10.0, 'penalty': 'l2', 'solver': 'lbfgs'},
                 {'C': 1.0, 'penalty': 'l1', 'solver': 'liblinear'},
                 {'C': 1.0, 'penalty': 'elasticnet', 'solver': 'saga', 'l1_ratio': 0.5}],
 'lr5_max_iter': 5000,
 'n_jobs': 1,
 'blas_threads': 1,
 'resume': True,
 'make_plots': True,
 'show_channel_names': False,
 'class_names': {0: 'Up', 1: 'Down', 2: 'Right', 3: 'Left'},
 'evaluation_scope': 'pooled_subject',
 'run_ml': True,
 'run_dl': True,
 'run_pretrained': True,
 'device': 'auto',
 'torch_threads': 2,
 'neural_sfreq': 200.0,
 'neural_bandpass': [2.0, 45.0],
 'save_neural_weights': False,
 'dl': {'widths': [32, 64],
        'head_width': 64,
        'max_epochs': 80,
        'patience': 12,
        'batch_size': 16,
        'lr': 0.001,
        'weight_decay': 0.01,
        'gradient_clip': 1.0,
        'print_every': 5},
 'pretrained': {'model': 'CBraMod',
                'modes': ['frozen_probe'],
                'checkpoint_repo': 'weighting666/CBraMod',
                'checkpoint_filename': 'pretrained_weights.pth',
                'checkpoint_revision': '500543c7e30bda1b22bfd51a49301b238dee21fd',
                'checkpoint_sha256': '0792cb808c14e6b7a2bb2ce1dff379bc47bc54c49a779825bdfeb33bf8157178',
                'checkpoint_path': None,
                'allow_download': True,
                'embedding_batch_size': 4,
                'probe_c': 1.0,
                'probe_max_iter': 3000,
                'head_width': 64,
                'dropout': 0.5,
                'max_epochs': 40,
                'patience': 8,
                'batch_size': 4,
                'lr': 0.001,
                'encoder_lr': 1e-05,
                'weight_decay': 0.01,
                'gradient_clip': 1.0,
                'print_every': 5}}


## 3. Embedded audited ML engine

The loader verifies FIF/event labels and timestamps, retains only INNER SPEECH, crops by the FIF clock, preserves original trial IDs, and checks 128 EEG channel names. The current extractor computes eight-window Burg AR4/db2-DWT entropy and log variance, plus filtered-power features. Features are channel-local and label-free.

After choosing a fixed channel set, supervised selection is rerun on that candidate pool. PAPER_ORDER selects on the complete subject (or complete session when within-session mode is selected). NESTED_SAFE selects on outer training rows and, when tuning is enabled, on each inner training subset. Classifier scaling always uses training rows. K is never silently clipped. Caches include source hashes, ordered rows, labels, channel membership, settings and engine version.


In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                    separators=(',', ':')).encode()).hexdigest()

def array_digest(value):
    value = np.ascontiguousarray(value)
    h = hashlib.sha256(str((value.shape, value.dtype.str)).encode())
    h.update(memoryview(value).cast('B'))
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def bandpower_features(X, sfreq):
    n = min(int(round(sfreq)), X.shape[-1])
    if sfreq / 2 < 45 or n < 16:
        raise ValueError('Insufficient sampling rate or epoch length for declared bands.')
    freqs, density = welch(X, fs=sfreq, window='hann', nperseg=n,
                          noverlap=n//2, detrend='constant', scaling='density', axis=-1)
    values = []
    for low, high in BANDS:
        mask = (freqs >= low) & (freqs <= high)
        if mask.sum() < 2:
            raise ValueError('Insufficient frequency bins.')
        power = np.trapezoid(density[...,mask], freqs[mask], axis=-1)
        values.append(np.log10(np.maximum(power, 1e-24)))
    return np.stack(values, axis=-1).reshape(len(X), -1).astype(np.float64)

def burg_coefficients(data, order=4):
    x = np.asarray(data, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Burg requires finite [signals, samples].')
    if not isinstance(order, (int,np.integer)) or not 1 <= order < x.shape[1]-1:
        raise ValueError('AR order must be positive and smaller than samples minus one.')
    x = x-x.mean(axis=-1,keepdims=True)
    scale = np.sqrt(np.mean(x*x,axis=-1,keepdims=True))
    x = x/np.maximum(scale,1e-15)
    forward,backward = x[:,1:].copy(),x[:,:-1].copy()
    coefficients = np.empty((len(x),0),dtype=np.float64)
    for lag in range(order):
        denominator = np.sum(forward*forward+backward*backward,axis=-1)
        reflection = np.divide(-2*np.sum(forward*backward,axis=-1),denominator,
            out=np.zeros(len(x)),where=denominator>1e-15)
        reflection = np.clip(reflection,-1+1e-7,1-1e-7)
        coefficients = np.concatenate([
            coefficients+reflection[:,None]*coefficients[:,::-1],reflection[:,None]],axis=1)
        updated_forward = forward+reflection[:,None]*backward
        updated_backward = backward+reflection[:,None]*forward
        forward,backward = updated_forward[:,1:],updated_backward[:,:-1]
    return -coefficients

def wavelet_statistics(coefficients):
    values = np.asarray(coefficients,dtype=np.float64)
    energy = values*values
    total = energy.sum(axis=-1,keepdims=True)
    probability = np.divide(energy,total,out=np.zeros_like(energy),where=total>1e-24)
    entropy = -np.sum(probability*np.log(np.maximum(probability,1e-300)),axis=-1)
    entropy = entropy/np.log(values.shape[-1]) if values.shape[-1]>1 else np.zeros_like(entropy)
    variance = np.log(np.maximum(np.var(values,axis=-1,ddof=0),1e-12))
    return np.clip(entropy,0.,1.),variance

def extract_trial_features(X,sfreq,ch_names,config):
    feature_set = config.get('feature_set','bandpower')
    rows = []
    def add(channel,window,family,component,start,stop):
        rows.append(dict(feature_index=len(rows),channel_index=int(channel),channel=ch_names[channel],
            window=int(window),family=family,component=component,
            start_time_mne=float(config['action_window_mne'][0]+start),
            stop_time_mne=float(config['action_window_mne'][0]+stop),
            feature_name=f'w{window:02d}__{ch_names[channel]}__{family}__{component}'))
    if feature_set=='bandpower':
        F = bandpower_features(X,sfreq)
        for channel in range(len(ch_names)):
            for lo,hi in BANDS:
                add(channel,0,'welch_log10_power',f'{lo:g}-{hi:g}Hz',0,X.shape[-1]/sfreq)
        return F,pd.DataFrame(rows),dict(feature_set=feature_set,units='volts',bands=BANDS,
            features=int(F.shape[1]),one_row_per_original_trial=True)
    if feature_set not in ['eight_window','combined']:
        raise ValueError('FEATURE_SET must be bandpower, eight_window, or combined.')
    import pywt
    n_windows = config.get('n_windows',8)
    wavelet = config.get('wavelet','db2')
    level = config.get('wavelet_level',3)
    ar_order = config.get('ar_order',4)
    target_rate = config.get('feature_sfreq',128.)
    lo,hi = config.get('wavelet_bandpass',[2.,45.])
    if n_windows!=8:
        raise ValueError('This reference implementation uses exactly eight windows.')
    if not 0 < lo < hi < min(sfreq,target_rate)/2:
        raise ValueError('Wavelet bandpass must be below both sampling Nyquist frequencies.')
    ratio = Fraction(float(target_rate)/float(sfreq)).limit_denominator(10000)
    samples = (X.shape[-1]*ratio.numerator+ratio.denominator-1)//ratio.denominator
    minimum_window = samples//n_windows
    maximum = pywt.dwt_max_level(minimum_window,wavelet)
    if not isinstance(level,(int,np.integer)) or not 1<=level<=maximum:
        raise ValueError(f'{wavelet} level {level} exceeds maximum {maximum} for {minimum_window}-sample windows.')
    if not isinstance(ar_order,(int,np.integer)) or not 1<=ar_order<minimum_window-1:
        raise ValueError('AR_ORDER is invalid for the window length.')
    microvolts = np.asarray(X,dtype=np.float32)*1e6
    sos = signal.butter(4,(lo,hi),btype='bandpass',fs=sfreq,output='sos')
    broadband = signal.sosfiltfilt(sos,microvolts,axis=-1).astype(np.float32)
    broadband = signal.resample_poly(broadband,ratio.numerator,ratio.denominator,axis=-1)
    components = [f'A{level}']+[f'D{i}' for i in range(level,0,-1)]
    descriptors = [('ar',f'lag{i}') for i in range(1,ar_order+1)]
    descriptors += [('wavelet_entropy',c) for c in components]
    descriptors += [('wavelet_log_variance',c) for c in components]
    windows = np.array_split(broadband,n_windows,axis=-1)
    matrices=[]
    offset=0
    for window,values in enumerate(windows,1):
        signals = values.reshape(-1,values.shape[-1])
        ar = burg_coefficients(signals,ar_order)
        coefficients = pywt.wavedec(signals.astype(np.float64),wavelet,
                                    mode='symmetric',level=level,axis=-1)
        summaries = [wavelet_statistics(c) for c in coefficients]
        entropy = np.stack([s[0] for s in summaries],axis=-1)
        variance = np.stack([s[1] for s in summaries],axis=-1)
        descriptor_values = np.concatenate([ar,entropy,variance],axis=-1)
        matrices.append(descriptor_values.reshape(len(X),-1))
        for channel in range(len(ch_names)):
            for family,component in descriptors:
                add(channel,window,family,component,offset/target_rate,(offset+values.shape[-1])/target_rate)
        offset += values.shape[-1]
    F = np.concatenate(matrices,axis=1)
    if feature_set=='combined':
        # Reference fusion: six filtered bands, whole epoch + three contiguous temporal bins.
        fusion_bands = ((2.,4.),(4.,8.),(8.,13.),(13.,20.),(20.,30.),(30.,45.))
        filtered = np.stack([signal.sosfiltfilt(signal.butter(4,band,btype='bandpass',fs=sfreq,output='sos'),
                                              microvolts,axis=-1).astype(np.float32) for band in fusion_bands],axis=1)
        edges = np.linspace(0,X.shape[-1],4).astype(int)
        bounds = [(0,X.shape[-1])]+list(zip(edges[:-1],edges[1:]))
        power = []
        for temporal,(start,stop) in enumerate(bounds):
            power.append(np.log(np.maximum(np.mean(filtered[...,start:stop]**2,axis=-1),1e-12)).reshape(len(X),-1))
            for low,high in fusion_bands:
                for channel in range(len(ch_names)):
                    add(channel,temporal,'filtered_log_power',f'{low:g}-{high:g}Hz',start/sfreq,stop/sfreq)
        F = np.concatenate([F,*power],axis=1)
    F = F.astype(np.float32)
    schema = pd.DataFrame(rows)
    if F.shape!=(len(X),len(schema)) or not np.isfinite(F).all() or schema.feature_name.duplicated().any():
        raise ValueError('Invalid descriptor values or schema.')
    info = dict(feature_set=feature_set,n_windows=n_windows,wavelet=wavelet,wavelet_level=level,
        wavelet_components=components,wavelet_mode='symmetric',boundary_coefficients='retained',
        ar_order=ar_order,ar_method='univariate_burg_demeaned',bandpass=[lo,hi],input_units='microvolts',
        input_sfreq=sfreq,feature_sfreq=target_rate,window_lengths=[w.shape[-1] for w in windows],
        entropy='normalized_negative_sum_p_log_p',variance='natural_log_population_variance_floor_1e-12',
        features=F.shape[1],one_row_per_original_trial=True,paper_exact_reproduction=False)
    return F,schema,info


In [ ]:
def safe_ratio(a, b):
    return a / np.maximum(np.abs(b), 1e-12)

def statistics25(x):
    """25 defined temporal descriptors, vectorized over leading dimensions."""
    x = np.asarray(x, np.float64)
    d = np.diff(x, axis=-1)
    dd = np.diff(d, axis=-1)
    mean, std = x.mean(axis=-1), x.std(axis=-1)
    absolute = np.abs(x)
    rms = np.sqrt(np.mean(x*x, axis=-1))
    variance = std**2
    mobility = np.sqrt(safe_ratio(np.var(d, axis=-1), variance))
    complexity = safe_ratio(np.sqrt(safe_ratio(np.var(dd, axis=-1), np.var(d, axis=-1))), mobility)
    # Central moments are explicitly zero for constant signals.
    standardized = (x - mean[..., None]) / np.maximum(std[..., None], 1e-12)
    values = [mean, std, variance, np.min(x, axis=-1), np.max(x, axis=-1),
              np.ptp(x, axis=-1), np.median(x, axis=-1),
              np.percentile(x, 75, axis=-1)-np.percentile(x, 25, axis=-1),
              absolute.mean(axis=-1), rms, np.sum(x*x, axis=-1),
              np.mean(standardized**3, axis=-1),
              np.where(std > 1e-12, np.mean(standardized**4, axis=-1)-3, 0),
              np.mean(np.abs(x-mean[..., None]), axis=-1),
              np.median(np.abs(x-np.median(x, axis=-1)[..., None]), axis=-1),
              np.mean(np.diff(np.signbit(x), axis=-1), axis=-1),
              np.sum(np.abs(d), axis=-1), np.mean(np.abs(d), axis=-1),
              np.mean(d*d, axis=-1), mobility, complexity,
              safe_ratio(np.max(absolute, axis=-1), rms),
              safe_ratio(rms, absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), np.mean(np.sqrt(absolute), axis=-1)**2)]
    names = ["mean", "std", "variance", "min", "max", "range", "median", "iqr",
             "mean_absolute", "rms", "energy", "skewness", "excess_kurtosis",
             "mean_absolute_deviation", "median_absolute_deviation", "zero_crossing_rate",
             "waveform_length", "mean_abs_difference", "mean_squared_difference",
             "hjorth_mobility", "hjorth_complexity", "crest_factor", "shape_factor",
             "impulse_factor", "clearance_factor"]
    return np.stack(values, axis=-1), names

def frequency24(x, sfreq, high):
    freq, psd = signal.welch(x, fs=sfreq, nperseg=min(256, x.shape[-1]), axis=-1)
    keep = (freq >= 0.5) & (freq <= high)
    f, p = freq[keep], psd[..., keep]
    if len(f) < 3:
        raise ValueError("Too few frequency bins")
    prob = safe_ratio(p, p.sum(axis=-1)[..., None])
    center = (prob*f).sum(axis=-1)
    width = np.sqrt((prob*(f-center[..., None])**2).sum(axis=-1))
    cum = np.cumsum(prob, axis=-1)
    roll = lambda q: f[np.argmax(cum >= q, axis=-1)]
    entropy = -np.sum(prob*np.log(np.maximum(prob, 1e-300)), axis=-1)/np.log(len(f))
    flatness = safe_ratio(np.exp(np.log(np.maximum(p, 1e-300)).mean(axis=-1)), p.mean(axis=-1))
    peak = p.max(axis=-1)
    weighted_std = np.maximum(width, 1e-12)[..., None]
    centered_power = p-p.mean(axis=-1)[..., None]
    sd_power = np.maximum(p.std(axis=-1), 1e-12)[..., None]
    power_z = centered_power/sd_power
    values = [trapezoid(p, f, axis=-1), f[p.argmax(axis=-1)], center, width,
              roll(0.5), roll(0.85), roll(0.95), entropy, flatness,
              safe_ratio(peak, p.mean(axis=-1)),
              (prob*((f-center[..., None])/weighted_std)**3).sum(axis=-1),
              (prob*((f-center[..., None])/weighted_std)**4).sum(axis=-1),
              np.mean(power_z**3, axis=-1), np.mean(power_z**4, axis=-1)-3]
    names = ["total_power", "peak_frequency", "centroid", "bandwidth", "median_frequency",
             "rolloff85", "rolloff95", "entropy", "flatness", "crest",
             "frequency_skewness", "frequency_kurtosis", "psd_skewness", "psd_excess_kurtosis"]
    total = trapezoid(p, f, axis=-1)
    for name, lo, hi in [("delta", .5, 4), ("theta", 4, 8), ("alpha", 8, 13),
                         ("beta", 13, 30), ("gamma", 30, high)]:
        mask = (freq >= lo) & (freq < hi if hi < high else freq <= hi)
        power = trapezoid(psd[..., mask], freq[mask], axis=-1)
        values.extend([power, safe_ratio(power, total)])
        names.extend([f"{name}_power", f"{name}_relative_power"])
    return np.stack(values, axis=-1), names

def wavelet_features(x, wavelet, level):
    if pywt.dwt_max_level(x.shape[-1], pywt.Wavelet(wavelet).dec_len) < level:
        raise ValueError("Window is too short for the requested DWT level")
    coefficients = pywt.wavedec(x, wavelet, level=level, axis=-1, mode="symmetric")
    labels = [f"A{level}"] + [f"D{i}" for i in range(level, 0, -1)]
    values, names = [], []
    for band, c in zip(labels, coefficients):
        d = np.diff(c, axis=-1)
        std = c.std(axis=-1)
        z = (c-c.mean(axis=-1)[..., None])/np.maximum(std[..., None], 1e-12)
        probability = safe_ratio(c*c, np.sum(c*c, axis=-1)[..., None])
        # Geometric length of amplitude-normalized coefficients; definition explicit
        # because the paper's "fractal length" formula is unavailable.
        norm = safe_ratio(c-c.min(axis=-1)[..., None], np.ptp(c, axis=-1)[..., None])
        geometric_length = np.sum(np.sqrt(np.diff(norm, axis=-1)**2 + (1/max(c.shape[-1]-1, 1))**2), axis=-1)
        band_values = [c.mean(axis=-1), np.mean(np.abs(c), axis=-1), std, std**2,
                       np.sqrt(np.mean(c*c, axis=-1)), np.mean(c*c, axis=-1),
                       np.sum(c*c, axis=-1), np.sum(np.abs(d), axis=-1),
                       np.ptp(c, axis=-1), np.mean(z**3, axis=-1),
                       np.where(std > 1e-12, np.mean(z**4, axis=-1)-3, 0),
                       -np.sum(probability*np.log(np.maximum(probability, 1e-300)), axis=-1),
                       geometric_length]
        band_names = ["mean", "mean_absolute", "std", "variance", "rms", "mean_power",
                      "energy", "waveform_length", "range", "skewness", "excess_kurtosis",
                      "energy_entropy", "normalized_geometric_length"]
        values.extend(band_values)
        names.extend([f"{band}_{n}" for n in band_names])
    return np.stack(values, axis=-1), names

def documented_features(trial, cfg):
    """50 TD + 24 FD + 78 TFD = 152/channel; deliberately not padded to 191."""
    td, td_names = statistics25(trial)
    env, env_names = statistics25(np.abs(signal.hilbert(trial, axis=-1)))
    fd, fd_names = frequency24(trial, cfg.expected_sfreq, cfg.spectral_high)
    tf, tf_names = wavelet_features(trial, cfg.wavelet, cfg.wavelet_level)
    matrix = np.concatenate([td, env, fd, tf], axis=-1)
    names = ([f"TD_signal_{n}" for n in td_names] + [f"TD_envelope_{n}" for n in env_names]
             + [f"FD_{n}" for n in fd_names] + [f"TFD_{n}" for n in tf_names])
    if matrix.shape[-1] != 152 or len(set(names)) != 152:
        raise AssertionError("Feature schema changed")
    if not np.isfinite(matrix).all():
        raise ValueError("Non-finite features; do not silently replace numerical failures")
    return matrix, names


In [ ]:
def environment_info():
    result = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                  scipy=scipy.__version__, sklearn=sklearn.__version__)
    for name in ('mne', 'PyWavelets', 'lightgbm', 'vmdpy'):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = 'not installed'
    return result

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.tmp')
    with temp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    temp.replace(path)

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.csv.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)

def boundary_label(boundary):
    return PAPER_LABEL if boundary == 'PAPER_ORDER' else SAFE_LABEL

def discover_data_root(requested):
    if requested is not None:
        root = Path(requested)
        if not root.exists():
            raise FileNotFoundError(f'DATA_ROOT does not exist: {root}')
        return root
    base = Path('/kaggle/input')
    parents = {p.parent for p in base.rglob('sub-*_ses-*_eeg-epo.fif')} if base.exists() else set()
    if not parents:
        raise FileNotFoundError('No processed FIF files discovered. Set CONFIG["data_root"].')
    # Select the enclosing derivatives directory, never silently combine dataset copies.
    roots = {next((q for q in [p, *p.parents] if q.name == 'derivatives'), p) for p in parents}
    if len(roots) != 1:
        raise ValueError(f'Multiple dataset roots: {sorted(map(str, roots))}. Set data_root explicitly.')
    return roots.pop()

def discover_subjects(root):
    return sorted({int(re.match(r'sub-(\d+)_ses-', p.name)[1])
                   for p in Path(root).rglob('sub-*_ses-*_eeg-epo.fif')})

def vmd_backend():
    try:
        from vmdpy import VMD
        return VMD, 'vmdpy'
    except ImportError:
        try:
            from sktime.libs.vmdpy import VMD
            return VMD, 'sktime.libs.vmdpy'
        except ImportError:
            return None, 'unavailable'

def robust_z(x):
    med = np.median(x)
    return np.abs(x-med) / max(1.4826*np.median(np.abs(x-med)), 1e-12)

def apply_vmd(X, sfreq, names, subject, config):
    mode = config['vmd_mode']
    if mode == 'off':
        return X, dict(requested=mode, effective='off', backend='unused', applications=[])
    VMD, backend = vmd_backend()
    if VMD is None:
        print(f'Subject {subject:02d}: VMD requested but no installed backend; SKIPPED. No pip/internet used.', flush=True)
        return X, dict(requested=mode, effective='skipped_missing_backend', backend=backend, applications=[])
    cleaned = np.asarray(X, dtype=np.float64).copy()
    report = dict(requested=mode, effective=mode, backend=backend, applications=[],
                  approximation='Manual choices or robust per-trial channel mean/log-std detector; threshold and VMD parameters are assumptions.')
    manual = config['vmd_manual_channels'].get(subject, config['vmd_manual_channels'].get(str(subject), []))
    if any(name not in names for name in manual):
        raise ValueError(f'Unknown manual VMD channels: {set(manual)-set(names)}')
    for row, trial in enumerate(cleaned):
        if mode == 'manual':
            mask = np.array([name in manual for name in names])
        else:
            # Label-free, trial-local across-channel detector, no held-out-distribution fit.
            uv = trial*1e6
            mask = ((robust_z(uv.mean(axis=1)) > config['vmd_robust_z']) |
                    (robust_z(np.log(np.maximum(uv.std(axis=1), 1e-12))) > config['vmd_robust_z']))
        for col in np.flatnonzero(mask):
            x = trial[col]*1e6
            if np.std(x) < 1e-12:
                continue
            scale = max(float(np.sqrt(np.mean(x*x))), 1e-12)
            padded = np.pad(x/scale, (0, len(x)%2), mode='edge')
            p = config['vmd_parameters']
            modes, _, omega = VMD(padded, p['alpha'], p['tau'], 6, p['dc'], p['init'], p['tol'])
            if len(modes) != 6 or omega.ndim != 2:
                raise ValueError('Unexpected VMD output.')
            lowest = int(np.argmin(omega[-1]))
            clean = modes[np.arange(6) != lowest].sum(axis=0)[:len(x)] * scale / 1e6
            if not np.isfinite(clean).all():
                raise ValueError('Nonfinite VMD output.')
            trial[col] = clean
            report['applications'].append(dict(trial_row=row, channel=names[col], removed_mode=lowest,
                                               removed_hz=float(omega[-1, lowest]*sfreq)))
        if row == 0 or (row+1)%20 == 0:
            print(f'  VMD trial {row+1}/{len(X)}', flush=True)
    return cleaned.astype(np.float32), report

def feature_matrix(X, sfreq, names, mode, config):
    if mode == 'current':
        p = dict(config['current_features'], action_window_mne=config['action_window_mne'])
        F, schema, info = extract_trial_features(X, sfreq, names, p)
        schema = schema.copy()
        schema['original_feature_name'] = schema.feature_name
        schema['feature_name'] = schema.apply(lambda r: f'{r.channel}__w{r.window:02d}__{r.family}__{r.component}', axis=1)
        info['note'] = 'Current feature values and ordering reused exactly; names reformatted only.'
    else:
        cfg = SimpleNamespace(expected_sfreq=sfreq, **config['paper_inspired_features'])
        if cfg.wavelet_level != 5:
            raise ValueError('The existing documented_152 schema requires wavelet_level=5.')
        if not 30 < cfg.spectral_high <= sfreq/2:
            raise ValueError('spectral_high must be >30 and <= Nyquist.')
        # Existing approximation uses microvolts; explicit unit avoids scale-dependent numerical floors.
        values, feature_names = documented_features(X.astype(np.float64)*1e6, cfg)
        F = values.reshape(len(X), -1).astype(np.float32)
        schema = pd.DataFrame([dict(feature_index=i*len(feature_names)+j, channel_index=i,
                                   channel=channel, family=name.split('_')[0], component=name,
                                   feature_name=f'{channel}__{name}')
                              for i, channel in enumerate(names) for j, name in enumerate(feature_names)])
        info = dict(feature_mode=mode, features_per_channel=len(feature_names), units='microvolts',
                    wavelet=cfg.wavelet, wavelet_level=cfg.wavelet_level, spectral_high=cfg.spectral_high,
                    counts=dict(TD=50, FD=24, TFD=78), paper_exact_reproduction=False,
                    geometric_length='Amplitude-normalized polygonal length; not verified as the paper fractal-length formula.')
    if F.shape[1] != len(schema) or schema.feature_name.duplicated().any() or not np.isfinite(F).all():
        raise ValueError('Invalid feature schema or nonfinite values.')
    counts = schema.groupby('channel').size()
    if counts.nunique() != 1:
        raise ValueError('Feature counts differ across channels.')
    print(f'{"Paper-inspired feature set" if mode == "paper_inspired" else "Current feature set"}: '
          f'{int(counts.iloc[0])} features/channel; {F.shape[1]} total. Not an exact 191-feature replication.', flush=True)
    return F, schema, info

def prepare_subject_replication(subject, config):
    import mne
    root = Path(config['data_root'])
    files = sorted(root.rglob(f'sub-{subject:02d}_ses-*_eeg-epo.fif'))
    if config['sessions'] is not None:
        files = [p for p in files if int(re.search(r'_ses-(\d+)_', p.name)[1]) in config['sessions']]
    if not files:
        raise FileNotFoundError(f'No requested sessions for subject {subject}.')
    session_ids = [int(re.search(r'_ses-(\d+)_', p.name)[1]) for p in files]
    if len(set(session_ids)) != len(session_ids):
        raise ValueError('Duplicate subject/session FIF files; choose one dataset version.')
    if config['sessions'] is not None and set(session_ids) != set(config['sessions']):
        raise ValueError('Some requested sessions are missing.')
    paths = [q for p in files for q in (p, p.with_name(p.name.replace('_eeg-epo.fif','_events.dat')))]
    sources = [dict(path=str(p), sha256=file_sha256(p)) for p in paths]
    feature_config = {k: config[k] for k in ('action_window_mne', 'feature_mode', 'current_features',
                      'paper_inspired_features', 'vmd_mode', 'vmd_manual_channels', 'vmd_robust_z', 'vmd_parameters')}
    _, backend = vmd_backend() if config['vmd_mode'] != 'off' else (None, 'unused')
    key = digest(dict(subject=subject, sources=sources, feature_config=feature_config,
                      engine=ENGINE_ID, versions=environment_info(), vmd_backend=backend))
    cache = Path(config['output_root'])/'feature_cache'/key
    if config['resume'] and (cache/'manifest.json').exists():
        manifest = json.loads((cache/'manifest.json').read_text())
        F = np.load(cache/'features.npz')['F']
        metadata = pd.read_csv(cache/'metadata.csv')
        schema = pd.read_csv(cache/'schema.csv')
        if array_digest(F) != manifest['F_sha256'] or len(F) != len(metadata):
            raise ValueError('Feature cache integrity check failed.')
        print(f'Subject {subject:02d}: feature cache reused ({F.shape}); VMD: {manifest["vmd_effective"]}', flush=True)
        return dict(subject=subject, F=F, y=metadata.class_id.to_numpy(dtype=int), metadata=metadata,
                    schema=schema, manifest=manifest, feature_key=key)
    arrays, records, provenance, vmd_reports = [], [], [], []
    reference = None
    for path, session in zip(files, session_ids):
        event_path = path.with_name(path.name.replace('_eeg-epo.fif','_events.dat'))
        # Only the trusted dataset's event pickle is used, as in the audited existing loader.
        raw = np.asarray(pd.read_pickle(event_path))
        epochs = mne.read_epochs(path, preload=False, proj=False, verbose='ERROR')
        if raw.shape != (len(epochs),4) or not np.isfinite(raw).all() or not np.equal(raw,np.floor(raw)).all():
            raise ValueError(f'Invalid event dimensions or values: {event_path}')
        events = raw.astype(np.int64)
        if not np.array_equal(events[:,0],epochs.events[:,0]) or not np.array_equal(events[:,1]+31,epochs.events[:,2]):
            raise ValueError('FIF/event alignment failed.')
        if not np.isin(events[:,1],LABELS).all():
            raise ValueError('Unknown direction codes.')
        original_rows = np.asarray(epochs.selection)
        inner = np.flatnonzero(events[:,2] == 1)
        if not len(inner):
            raise ValueError('No INNER SPEECH trials in a requested session.')
        epochs = epochs[inner].load_data()
        epochs.pick(mne.pick_types(epochs.info,eeg=True,exclude=[]))
        names, sfreq = list(epochs.ch_names), float(epochs.info['sfreq'])
        if len(names) != 128:
            raise ValueError('This replication keeps all 128 EEG channels.')
        start, stop = config['action_window_mne']
        if start < epochs.times[0] or stop > epochs.times[-1]+1/sfreq+1e-9:
            raise ValueError('Requested action interval is outside the FIF epoch.')
        mask = (epochs.times >= start) & (epochs.times < stop)
        times = epochs.times[mask]
        if not len(times):
            raise ValueError('Empty action interval.')
        if reference is not None and (names != reference[0] or sfreq != reference[1] or not np.array_equal(times,reference[2])):
            raise ValueError('Sessions have different channel orders, rates or action times.')
        reference = names,sfreq,times
        X = epochs.get_data(copy=False)[:,:,mask].astype(np.float32)
        if not np.isfinite(X).all():
            raise ValueError('Nonfinite EEG.')
        X, report = apply_vmd(X,sfreq,names,subject,config)
        vmd_reports.append(dict(session=session, **report))
        arrays.append(X)
        for row, event in zip(original_rows[inner], events[inner]):
            records.append(dict(subject=subject,session=session,trial_index=int(row),cue_sample_raw=int(event[0]),
                                class_id=int(event[1]),condition='INNER',
                                trial_id=f'sub-{subject:02d}_ses-{session:02d}_row-{row:04d}'))
        provenance.append(dict(file=str(path),session=session,shape=list(X.shape),sfreq=sfreq,
                               first_time=float(times[0]),last_time=float(times[-1]),X_sha256=array_digest(X),
                               highpass=float(epochs.info['highpass']),lowpass=float(epochs.info['lowpass'])))
        print(f'Subject {subject:02d} session {session:02d}: X={X.shape}, sfreq={sfreq:g} Hz, INNER only.', flush=True)
    X = np.concatenate(arrays)
    metadata = pd.DataFrame(records)
    if metadata.trial_id.duplicated().any() or metadata.subject.nunique() != 1:
        raise ValueError('Trial IDs or subject separation failed.')
    F,schema,feature_info = feature_matrix(X,sfreq,names,config['feature_mode'],config)
    manifest = dict(subject=subject,pooled_shape=list(X.shape),sfreq=sfreq,channels=names,sources=sources,
                    provenance=provenance,feature_info=feature_info,feature_config=feature_config,
                    F_sha256=array_digest(F),vmd=vmd_reports,
                    vmd_effective=sorted({r['effective'] for r in vmd_reports}),feature_key=key,
                    action_interval_note='FIF epoch clock, audited cue-relative [0.5,3.0); original protocol action [1.0,3.5).')
    atomic_npz(cache/'features.npz',F=F)
    atomic_csv(metadata,cache/'metadata.csv')
    atomic_csv(schema,cache/'schema.csv')
    save_json(cache/'manifest.json',manifest)
    return dict(subject=subject,F=F,y=metadata.class_id.to_numpy(dtype=int),metadata=metadata,
                schema=schema,manifest=manifest,feature_key=key)

def dataset_summary(datasets, output_dir):
    rows = []
    for data in datasets.values():
        for session, meta in data['metadata'].groupby('session'):
            counts = meta.class_id.value_counts()
            rows.append(dict(subject=data['subject'],session=int(session),trials=len(meta),
                             trials_in_subject=len(data['y']),EEG_shape=str([len(meta),128,data['manifest']['pooled_shape'][2]]),
                             sfreq=data['manifest']['sfreq'],channels=128,
                             **{f'class_{i}':int(counts.get(i,0)) for i in LABELS}))
    table = pd.DataFrame(rows)
    atomic_csv(table,Path(output_dir)/'dataset_summary.csv')
    print('Subjects:', sorted(datasets), '\nSessions per subject:', table.groupby('subject').session.apply(list).to_dict())
    return table


In [ ]:
class MRMRQuotient(TransformerMixin, BaseEstimator):
    """Paper Eq.1 form: relevance / mean absolute correlation to selected features.

    MI and absolute Pearson correlation are explicit implementation choices.
    All candidates are considered; no relevance prefilter and no p-by-p matrix.
    """
    def __init__(self, k=590, relevance="mi", corr_floor=1e-6, seed=42):
        self.k, self.relevance, self.corr_floor, self.seed = k, relevance, corr_floor, seed

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        if len(X) != len(y) or not np.isfinite(X).all():
            raise ValueError("Invalid MRMR training input")
        self.n_features_in_ = X.shape[1]
        sd = X.std(axis=0)
        available_columns = np.flatnonzero(sd > 1e-12)
        if not len(available_columns):
            raise ValueError("All feature columns are constant")
        Z = (X[:, available_columns] - X[:, available_columns].mean(axis=0)) / sd[available_columns]
        if self.relevance == "mi":
            rel = mutual_info_classif(Z, y, random_state=self.seed)
        elif self.relevance == "f_stat":
            rel = f_classif(Z, y)[0]
        else:
            raise ValueError("Unknown MRMR relevance estimator")
        rel = np.nan_to_num(rel, nan=0, posinf=1e12, neginf=0)
        remaining = np.ones(len(rel), dtype=bool)
        redundancy_sum = np.zeros(len(rel), dtype=np.float64)
        chosen = []
        for step in range(min(int(self.k), len(rel))):
            score = rel.copy() if step == 0 else rel / np.maximum(redundancy_sum / step, self.corr_floor)
            score[~remaining] = -np.inf
            nxt = int(np.argmax(score))
            chosen.append(nxt)
            remaining[nxt] = False
            redundancy_sum += np.abs(Z.T @ Z[:, nxt] / len(Z))
        self.selected_indices_ = available_columns[np.asarray(chosen)]
        self.relevance_values_ = rel
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]


In [ ]:
def mutual_info_classif(X, y, random_state=42):
    """Same sklearn estimator, independent columns batched to expose progress."""
    parts = []
    for start in range(0, X.shape[1], 2048):
        stop = min(start+2048, X.shape[1])
        print(f'    MI relevance: features {start+1}–{stop}/{X.shape[1]}', flush=True)
        # Keep one RNG stream, as in a single call to sklearn.
        if start == 0:
            rng = np.random.RandomState(random_state)
        parts.append(_sklearn_mutual_info_classif(X[:,start:stop],y,random_state=rng))
    return np.concatenate(parts)

def cv_plan(data, config, scope='pooled_subject'):
    n = len(data['y'])
    units = [np.arange(n)] if scope == 'pooled_subject' else [g.index.to_numpy() for _,g in data['metadata'].groupby('session')]
    plan, visits = [], np.zeros(n,dtype=int)
    for unit in units:
        y = data['y'][unit]
        counts = np.bincount(y,minlength=4)
        if counts.min() < config['outer_folds']:
            raise ValueError(f'All four classes require >= {config["outer_folds"]} trials for this scope; counts={counts}. No silent fold reduction.')
        cv = StratifiedKFold(config['outer_folds'],shuffle=True,random_state=config['seed'])
        session = 0 if scope == 'pooled_subject' else int(data['metadata'].iloc[unit[0]].session)
        for fold,(a,b) in enumerate(cv.split(np.zeros(len(unit)),y),1):
            train,test = unit[a],unit[b]
            assert not np.intersect1d(train,test).size
            assert data['metadata'].iloc[train].subject.nunique() == 1
            if scope != 'pooled_subject':
                assert data['metadata'].iloc[train].session.nunique() == 1
            visits[test] += 1
            plan.append(dict(fold=fold,unit_session=session,train=train.tolist(),test=test.tolist(),selection_unit=unit.tolist()))
    assert np.all(visits==1), 'Exactly one held-out prediction per trial is required.'
    return plan

def ranking(data, fit_rows, selector, k_max, config, root):
    fit_rows = np.asarray(fit_rows,dtype=int)
    y = data['y'][fit_rows]
    key = digest(dict(feature_key=data['feature_key'],F=data['manifest']['F_sha256'],rows=fit_rows.tolist(),
                      y=array_digest(y),selector=selector,relevance=config['mrmr_relevance'],
                      corr_floor=config['mrmr_corr_floor'],seed=config['seed'],engine=ENGINE_ID,
                      versions=environment_info()))
    directory = Path(root)/'rank_cache'/key
    if config['resume'] and (directory/'ranking.npz').exists():
        with np.load(directory/'ranking.npz') as cached:
            indices = cached['indices']
        if len(indices) >= k_max:
            return indices[:k_max]
    F = data['F'][fit_rows].astype(np.float64)
    # Reused selector normalization is fitted at precisely fit_rows.
    eligible = np.flatnonzero(F.std(axis=0)>1e-12)
    if k_max > len(eligible):
        raise ValueError(f'K={k_max} exceeds {len(eligible)} eligible features; no silent clipping.')
    began = perf_counter()
    print(f'  Ranking {selector.upper()}: {len(fit_rows)} fit trials, {len(eligible)} candidates, K={k_max}; '
          f'relevance={config["mrmr_relevance"] if selector=="mrmr" else "f_stat"}', flush=True)
    with threadpool_limits(limits=config['blas_threads']):
        if selector == 'anova':
            values = np.nan_to_num(f_classif(F[:,eligible],y)[0],nan=0,posinf=1e12,neginf=0)
            indices = eligible[np.argsort(-values,kind='stable')[:k_max]]
        elif selector == 'mrmr':
            ranker = MRMRQuotient(k_max,config['mrmr_relevance'],config['mrmr_corr_floor'],config['seed'])
            ranker.fit(F,y)
            indices = ranker.selected_indices_
        else:
            raise ValueError('Unknown selector.')
    if len(indices)!=k_max or len(set(indices))!=k_max:
        raise ValueError('Selector did not return the requested unique feature count.')
    atomic_npz(directory/'ranking.npz',indices=indices)
    save_json(directory/'audit.json',dict(fit_rows=fit_rows.tolist(),fit_trial_ids=data['metadata'].iloc[fit_rows].trial_id.tolist(),
                                       labels_sha256=array_digest(y),selector=selector,k_max=k_max,eligible_features=len(eligible)))
    print(f'  Ranking done in {(perf_counter()-began)/60:.2f} minutes; prefixes reused for smaller K.', flush=True)
    return indices

class StratifiedLR5(ClassifierMixin, BaseEstimator):
    """Five configurable LR pipelines, stratified training-only bootstrap, mean probability vote."""
    def __init__(self, members=None, seed=42, max_iter=5000):
        self.members,self.seed,self.max_iter = members,seed,max_iter

    def fit(self, X, y):
        if self.members is None or len(self.members)!=5:
            raise ValueError('Exactly five LR member configurations are required.')
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        self.models_,self.bootstrap_indices_ = [],[]
        rng = np.random.default_rng(self.seed)
        for i, spec in enumerate(self.members):
            indices = np.concatenate([rng.choice(np.flatnonzero(y==c),np.sum(y==c),replace=True) for c in self.classes_])
            rng.shuffle(indices)
            params = dict(spec,max_iter=self.max_iter,random_state=self.seed+i)
            # Explicit OVR supports liblinear in multiclass tasks across sklearn versions.
            model = Pipeline([('scale',StandardScaler()),('lr',OneVsRestClassifier(LogisticRegression(**params),n_jobs=1))])
            model.fit(X[indices],y[indices])
            assert np.array_equal(model.classes_,self.classes_)
            self.models_.append(model)
            self.bootstrap_indices_.append(indices)
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_],axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X),axis=1)]

def make_model(name, config, override=None):
    seed = config['seed']
    params = dict(config['models'].get(name,{}))
    params.update(override or {})
    if name == 'svm':
        estimator,scale = SVC(**params),True
    elif name == 'lda':
        estimator,scale = LinearDiscriminantAnalysis(**params),True
    elif name == 'etc':
        estimator,scale = ExtraTreesClassifier(**params,random_state=seed,n_jobs=config['n_jobs']),False
    elif name == 'rf':
        estimator,scale = RandomForestClassifier(**params,random_state=seed,n_jobs=config['n_jobs']),False
    elif name == 'lr':
        estimator,scale = LogisticRegression(**params,random_state=seed),True
    elif name == 'lgbm':
        try:
            from lightgbm import LGBMClassifier
        except ImportError as exc:
            raise ImportError('LightGBM is unavailable; no substitute result is labeled LightGBM.') from exc
        estimator,scale = LGBMClassifier(**params,random_state=seed,n_jobs=config['n_jobs'],verbosity=-1),False
    elif name == 'lr5':
        return StratifiedLR5(config['lr5_members'],seed,config['lr5_max_iter'])
    else:
        raise ValueError(f'Unknown classifier: {name}')
    return Pipeline([('scale',StandardScaler() if scale else 'passthrough'),('model',estimator)])

def metrics(y, pred):
    rec = recall_score(y,pred,labels=LABELS,average=None,zero_division=0)
    return dict(accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                macro_f1=float(f1_score(y,pred,labels=LABELS,average='macro',zero_division=0)),
                **{f'recall_{i}':float(rec[i]) for i in LABELS})

def choose_inner(data, train, boundary, fixed_columns, selector, counts, config, root):
    counts = sorted(set(map(int,counts)))
    n_inner = config['inner_folds']
    if np.bincount(data['y'][train],minlength=4).min()<n_inner:
        raise ValueError('Insufficient outer-training examples for configured inner CV.')
    candidates = list(ParameterGrid(config['svm_inner_grid']))
    scores = np.zeros((len(counts),len(candidates),n_inner))
    cv = StratifiedKFold(n_inner,shuffle=True,random_state=config['seed'])
    audits = []
    for split,(a,b) in enumerate(cv.split(np.zeros(len(train)),data['y'][train])):
        inner_train,inner_valid = train[a],train[b]
        if boundary == 'NESTED_SAFE':
            columns = ranking(data,inner_train,selector,max(counts),config,root)
            selection_rows = inner_train
        else:
            columns = fixed_columns
            selection_rows = None # Pre-CV selection intentionally exposed these labels.
        assert not np.intersect1d(inner_train,inner_valid).size
        audits.append(dict(inner_fold=split+1,train=inner_train.tolist(),validation=inner_valid.tolist(),
                           selection_rows=None if selection_rows is None else selection_rows.tolist()))
        for ki,k in enumerate(counts):
            for pi,params in enumerate(candidates):
                model = make_model('svm',config,params)
                with threadpool_limits(limits=config['blas_threads']):
                    model.fit(data['F'][inner_train][:,columns[:k]],data['y'][inner_train])
                    p = model.predict(data['F'][inner_valid][:,columns[:k]])
                scores[ki,pi,split] = balanced_accuracy_score(data['y'][inner_valid],p)
    ki,pi = np.unravel_index(np.argmax(scores.mean(axis=2)),scores.shape[:2])
    return counts[ki],candidates[pi],dict(inner_splits=audits,candidates=candidates,k_values=counts,
                                       mean_balanced_accuracy=scores.mean(axis=2).tolist(),selected_from='inner validation only')

def evaluate(data, spec, boundary, config, root, scope='pooled_subject'):
    selector, requested_k, classifier = spec['selector'],spec['k'],spec['classifier']
    plan = cv_plan(data,config,scope)
    setting = dict(subject=data['subject'],feature_mode=config['feature_mode'],scope=scope,boundary=boundary,
                   selection_boundary=boundary_label(boundary),**spec)
    result_key = digest(dict(setting=setting,feature_key=data['feature_key'],plan=plan,
                             evaluation_config={k:config[k] for k in ('seed','outer_folds','inner_folds','mrmr_relevance',
                             'mrmr_corr_floor','models','lr5_members','lr5_max_iter','tune_svm','svm_inner_grid','k_values')},
                             engine=ENGINE_ID,versions=environment_info()))
    directory = Path(root)/'runs'/result_key
    directory.mkdir(parents=True,exist_ok=True)
    k_max = max(config['k_values']) if requested_k == 'inner' else int(requested_k)
    n = len(data['y'])
    predictions = np.full(n,-1,dtype=int)
    probabilities = np.full((n,4),np.nan)
    outer_fold,unit_session,visits = np.zeros(n,dtype=int),np.zeros(n,dtype=int),np.zeros(n,dtype=int)
    selections,fold_rows,audits = [],[],[]
    save_json(directory/'setting.json',dict(**setting,result_id=result_key,feature_key=data['feature_key']))
    save_json(directory/'plan.json',plan)
    for fold in plan:
        train,test = np.asarray(fold['train']),np.asarray(fold['test'])
        selection_rows = np.asarray(fold['selection_unit']) if boundary=='PAPER_ORDER' else train
        token = f"ses-{fold['unit_session']:02d}_fold-{fold['fold']:02d}"
        checkpoint = directory/f'{token}.npz'
        audit_path = directory/f'{token}.json'
        if config['resume'] and checkpoint.exists() and audit_path.exists():
            with np.load(checkpoint) as saved:
                pred,cols,prob = saved['pred'],saved['columns'],saved['probabilities']
            audit = json.loads(audit_path.read_text())
            if audit['test_rows'] != test.tolist() or audit['train_rows'] != train.tolist() or len(pred)!=len(test):
                raise ValueError('Checkpoint split mismatch.')
            chosen_k = audit['chosen_k']
            print(f"  Subject {data['subject']:02d}, fold {fold['fold']:02d}: resumed {boundary_label(boundary)}",flush=True)
        else:
            start = perf_counter()
            print(f"Subject {data['subject']:02d}, {scope}, {selector.upper()}, K={requested_k}, {classifier.upper()}, "
                  f"{boundary_label(boundary)}, fold {fold['fold']}/{config['outer_folds']} (session unit {fold['unit_session']}).",flush=True)
            cols = ranking(data,selection_rows,selector,k_max,config,root)
            chosen_k,params,inner_audit = k_max,{},None
            if classifier=='svm' and (config['tune_svm'] or requested_k=='inner'):
                counts = config['k_values'] if requested_k=='inner' else [k_max]
                chosen_k,params,inner_audit = choose_inner(data,train,boundary,cols,selector,counts,config,root)
            cols = cols[:chosen_k]
            if boundary=='NESTED_SAFE':
                assert not np.intersect1d(selection_rows,test).size
            model = make_model(classifier,config,params)
            with threadpool_limits(limits=config['blas_threads']):
                model.fit(data['F'][train][:,cols],data['y'][train])
                pred = model.predict(data['F'][test][:,cols])
                train_pred = model.predict(data['F'][train][:,cols])
                prob = model.predict_proba(data['F'][test][:,cols]) if hasattr(model,'predict_proba') else np.empty((len(test),0))
            bootstrap = []
            if classifier=='lr5':
                bootstrap = [train[x].tolist() for x in model.bootstrap_indices_]
                assert all(set(x).issubset(set(train)) and not set(x).intersection(test) for x in bootstrap)
            audit = dict(train_rows=train.tolist(),test_rows=test.tolist(),selection_fit_rows=selection_rows.tolist(),
                         scaler_fit_rows=train.tolist(),bootstrap_global_rows=bootstrap,
                         bootstrap_scalers_fit_on='each bootstrap sample' if classifier=='lr5' else 'outer train',
                         chosen_k=int(chosen_k),model_overrides=params,inner_selection=inner_audit,
                         train_scores=metrics(data['y'][train],train_pred),elapsed_seconds=perf_counter()-start,
                         selection_boundary=boundary_label(boundary))
            atomic_npz(checkpoint,pred=pred,columns=cols,probabilities=prob)
            save_json(audit_path,audit)
            print(f"  Held-out BA={balanced_accuracy_score(data['y'][test],pred):.3f}; fold finished in {audit['elapsed_seconds']/60:.2f} minutes.",flush=True)
        predictions[test] = pred
        if prob.shape == (len(test),4):
            probabilities[test] = prob
        outer_fold[test] = fold['fold']
        unit_session[test] = fold['unit_session']
        visits[test] += 1
        if boundary=='NESTED_SAFE':
            assert not set(audit['selection_fit_rows']).intersection(test)
        assert set(audit['scaler_fit_rows']).issubset(train)
        assert all(set(x).issubset(train) for x in audit['bootstrap_global_rows'])
        fold_rows.append(dict(**setting,fold=fold['fold'],unit_session=fold['unit_session'],chosen_k=chosen_k,
                              train_accuracy=audit['train_scores']['accuracy'],train_balanced_accuracy=audit['train_scores']['balanced_accuracy'],
                              **metrics(data['y'][test],pred)))
        selections.extend([dict(result_id=result_key,fold=fold['fold'],unit_session=fold['unit_session'],rank=i+1,
                                feature_index=int(col),feature_name=data['schema'].iloc[col].feature_name,
                                channel=data['schema'].iloc[col].channel,**setting) for i,col in enumerate(cols)])
        audits.append(audit)
    assert np.all(visits==1) and np.isin(predictions,LABELS).all()
    assert data['metadata'].subject.nunique()==1
    oof = data['metadata'].copy()
    oof['y_true'],oof['y_pred'],oof['outer_fold'],oof['unit_session'] = data['y'],predictions,outer_fold,unit_session
    if np.isfinite(probabilities).all():
        for label in LABELS:
            oof[f'probability_{label}'] = probabilities[:,label]
    for name,value in setting.items():
        oof[name]=value
    row = dict(**setting,result_id=result_key,n_trials=n,OOF_predictions_per_trial=1,
               vmd_effective=str(data['manifest']['vmd_effective']),**metrics(data['y'],predictions))
    selection_frame = pd.DataFrame(selections)
    atomic_csv(oof,directory/'oof.csv')
    atomic_csv(selection_frame,directory/'selected_features.csv')
    atomic_csv(pd.DataFrame(fold_rows),directory/'fold_scores.csv')
    atomic_csv(pd.DataFrame(confusion_matrix(data['y'],predictions,labels=LABELS),columns=[f'pred_{i}' for i in LABELS])
               .assign(true_class=LABELS),directory/'confusion.csv')
    save_json(directory/'result.json',row)
    return dict(row=row,oof=oof,selections=selection_frame,fold_scores=pd.DataFrame(fold_rows),directory=directory,audits=audits)


In [ ]:
"""Fixed spatial channel sets and paired evaluation-boundary comparisons."""
from copy import deepcopy
CHANNEL_EXTENSION_VERSION='channel-reduction-dual-1.0'
CHANNEL_MODEL_NAMES={'svm','lda','etc','rf','lr','lgbm','lr5'}
CHANNEL_BOUNDARY_COLORS={'PAPER_ORDER':'#d95f02','NESTED_SAFE':'#1b9e77'}


In [ ]:
def validate_channel_config(config):
    for key in ('subjects','channel_counts','boundaries','selectors','classifiers'):
        value=config[key]
        if key=='subjects' and value is None:
            continue
        if not value or len(value)!=len(set(value)):
            raise ValueError(f'{key} must be a nonempty list of unique values, or subjects=None.')
    if 128 not in config['channel_counts'] or any(not isinstance(n,int) or not 1<=n<=128 for n in config['channel_counts']):
        raise ValueError('Include the 128-channel reference; other channel counts must be integers1–128.')
    if set(config['boundaries'])!={'PAPER_ORDER','NESTED_SAFE'}:
        raise ValueError('This paired study requires both PAPER_ORDER and NESTED_SAFE.')
    if not set(config['classifiers']).issubset(CHANNEL_MODEL_NAMES):
        raise ValueError(f'Classifiers must be chosen from {sorted(CHANNEL_MODEL_NAMES)}.')
    if not set(config['selectors']).issubset({'anova','mrmr'}):
        raise ValueError('Selectors: anova and/or mrmr.')
    if config['outer_folds']!=10 or config['inner_folds']<2:
        raise ValueError('Use ten outer folds and at least two inner folds.')
    if not isinstance(config['k_features'],int) or config['k_features']<1:
        raise ValueError('k_features must be a positive integer.')
    if config['feature_mode'] not in {'current','paper_inspired'}:
        raise ValueError('feature_mode must be current or paper_inspired.')
    if config['mrmr_relevance'] not in {'mi','f_stat'}:
        raise ValueError('mrmr_relevance must be mi or f_stat.')
    if config['channel_method'] not in {'spatial','manual'}:
        raise ValueError('channel_method must be spatial or manual.')
    if config['vmd_mode']!='off':
        raise ValueError('This controlled experiment freezes VMD off; a VMD comparison requires a separate protocol.')
    if config['run_permutation_controls']:
        for key in ('permutation_channel_counts','permutation_classifiers','permutation_selectors'):
            if not config[key] or len(config[key])!=len(set(config[key])):
                raise ValueError(f'{key} must be a nonempty list of unique values.')
        if not isinstance(config['n_permutations'],int) or config['n_permutations']<1:
            raise ValueError('n_permutations must be positive.')
        if not config['permutation_channel_counts'] or not set(config['permutation_channel_counts']).issubset(config['channel_counts']):
            raise ValueError('permutation_channel_counts must be a nonempty subset of channel_counts.')
        if not set(config['permutation_classifiers']).issubset(CHANNEL_MODEL_NAMES):
            raise ValueError('Unknown permutation classifier.')
        if not config['permutation_selectors'] or not set(config['permutation_selectors']).issubset({'anova','mrmr'}):
            raise ValueError('Unknown permutation selector.')
        if config['tune_svm']:
            raise ValueError('Freeze tune_svm=False for this initial randomized-label diagnostic.')
    if config['blas_threads']<1 or len(config['lr5_members'])!=5:
        raise ValueError('Use positive blas_threads and five LR ensemble members.')
    return config

def initialise_channel_study(config):
    config=deepcopy(config)
    validate_channel_config(config)
    config['data_root']=discover_data_root(config['data_root'])
    found=discover_subjects(config['data_root'])
    config['subjects']=found if config['subjects'] is None else config['subjects']
    if not config['subjects'] or not set(config['subjects']).issubset(found):
        raise ValueError(f'Requested subjects missing. Available subjects: {found}')
    control_subjects=config['subjects'] if config['permutation_subjects'] is None else config['permutation_subjects']
    if config['run_permutation_controls'] and (not control_subjects or not set(control_subjects).issubset(config['subjects'])):
        raise ValueError('permutation_subjects must be available in this run; None uses all selected subjects.')
    config['permutation_subjects']=control_subjects
    config['k_values']=[config['k_features']] # FixedK; inner search, when enabled, tunes SVM parameters only.
    np.random.seed(config['seed'])
    root=Path(config['output_root'])/('experiment_'+digest(dict(config=config,engine=ENGINE_ID))[:16])
    root.mkdir(parents=True,exist_ok=True)
    # Shared numerical caches survive changes to plotting/stage switches; reports remain configuration-specific.
    cache_root=Path(config['output_root'])/'evaluation_cache'/ENGINE_ID[:16]
    cache_root.mkdir(parents=True,exist_ok=True)
    save_json(root/'configuration.json',dict(config=config,engine_id=ENGINE_ID,versions=environment_info(),
              source_inspection=SOURCE_INSPECTION,channel_extension_version=CHANNEL_EXTENSION_VERSION,
              note='Processed128-channel derivatives subset in software; no re-acquisition or low-density ICA is simulated.'))
    print('Subjects:',config['subjects'],'| channels:',config['channel_counts'],'| K:',config['k_features'])
    print('Models:',config['classifiers'],'| selectors:',config['selectors'])
    print('Main:',config['run_main'],'| permutations:',config['run_permutation_controls'],config['n_permutations'])
    print('Feature mode:',config['feature_mode'],'| MRMR relevance:',config['mrmr_relevance'],'| VMD:off')
    print('Outputs:',root)
    print('PAPER_ORDER uses held-out labels in supervised selection: intentionally leaky diagnostic.')
    return config,root,cache_root

def fixed_channel_sets(datasets,config,root):
    import mne
    montage=mne.channels.make_standard_montage('biosemi128')
    canonical=list(montage.ch_names)
    positions=montage.get_positions()['ch_pos']
    if len(canonical)!=128:
        raise ValueError('Unexpected BioSemi128 montage.')
    for subject,data in datasets.items():
        if set(data['manifest']['channels'])!=set(canonical):
            raise ValueError(f'Subject{subject} channel names do not match BioSemi128. No index-based guess is used.')
        if set(data['schema'].channel)!=set(canonical):
            raise ValueError('Feature schema/channel names do not match.')
    points=np.asarray([positions[name] for name in canonical])
    # Same inspected farthest-point geometry rule, evaluated once in canonical montage order.
    start=int(np.linalg.norm(points-[0,0,points[:,2].max()],axis=1).argmin())
    chosen,remaining=[start],set(range(128))-{start}
    while remaining:
        candidates=np.asarray(sorted(remaining))
        distances=np.linalg.norm(points[candidates,None]-points[chosen][None],axis=2).min(axis=1)
        nxt=int(candidates[np.argmax(distances)])
        chosen.append(nxt)
        remaining.remove(nxt)
    counts=sorted(config['channel_counts'])
    sets={}
    for count in counts:
        if config['channel_method']=='spatial':
            membership={canonical[i] for i in chosen[:count]}
        else:
            value=config['manual_channel_sets'].get(count,config['manual_channel_sets'].get(str(count)))
            if value is None or len(value)!=count or len(set(value))!=count or not set(value).issubset(canonical):
                raise ValueError(f'manual_channel_sets[{count}] needs {count} unique valid channel names.')
            membership=set(value)
        # Canonical ordering ensures reproducible names and geometry-independent array order.
        sets[count]=[name for name in canonical if name in membership]
    for small,large in zip(counts,counts[1:]):
        if not set(sets[small]).issubset(sets[large]):
            raise ValueError('Channel configurations must be nested subsets.')
    payload=dict(method=config['channel_method'],sets={str(k):v for k,v in sets.items()},
                 canonical_order=canonical,greedy_spatial_order=[canonical[i] for i in chosen],
                 xyz_m={name:list(map(float,positions[name])) for name in canonical},
                 choice_basis='Predefined geometry/manual names only; no EEG values or labels.',
                 same_across_subjects_models_boundaries=True)
    payload['configuration_id']=digest(payload)
    save_json(Path(root)/'channel_configurations.json',payload)
    rows=[dict(channel_count=count,channel=name,canonical_index=canonical.index(name),
               x_m=float(positions[name][0]),y_m=float(positions[name][1]),z_m=float(positions[name][2]))
          for count,names in sets.items() for name in names]
    atomic_csv(pd.DataFrame(rows),Path(root)/'channel_configurations.csv')
    return sets,payload

def plot_channel_sets(channel_sets,geometry,root,show_names=False):
    positions=geometry['xyz_m']
    all_names=geometry['canonical_order']
    points=np.asarray([positions[name] for name in all_names])
    ncols=min(4,len(channel_sets))
    nrows=int(np.ceil(len(channel_sets)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(4*ncols,4*nrows),squeeze=False)
    for ax,(count,names) in zip(axes.flat,sorted(channel_sets.items(),reverse=True)):
        selected=np.array([positions[name] for name in names])
        ax.scatter(points[:,0],points[:,1],s=13,c='#dddddd')
        ax.scatter(selected[:,0],selected[:,1],s=30,c='#2878b5')
        if show_names:
            for name,(x,y,z) in zip(names,selected):
                ax.text(x,y,name,fontsize=5,ha='center',va='bottom')
        ax.set(title=f'{count} channels',xlabel='Left / Right (m)',ylabel='Posterior / Anterior (m)',aspect='equal')
        ax.grid(alpha=.15)
    for ax in list(axes.flat)[len(channel_sets):]:
        ax.axis('off')
    fig.suptitle('Fixed BioSemi 128 geometry — identical nested sets for subjects, models and boundaries')
    fig.tight_layout()
    fig.savefig(Path(root)/'channel_positions.png',dpi=160,bbox_inches='tight')
    plt.show()
    return fig

def channel_view(data,names,geometry):
    if not names or len(set(names))!=len(names):
        raise ValueError('Empty or repeated channel names.')
    mask=data['schema'].channel.isin(names).to_numpy()
    if set(data['schema'].loc[mask,'channel'])!=set(names):
        raise ValueError('Requested channel missing from feature schema.')
    indices=np.flatnonzero(mask)
    schema=data['schema'].iloc[indices].copy().reset_index(drop=True)
    schema['source_feature_index']=indices
    schema['feature_index']=np.arange(len(schema))
    manifest=deepcopy(data['manifest'])
    F=np.ascontiguousarray(data['F'][:,indices])
    manifest['parent_F_sha256']=data['manifest']['F_sha256']
    manifest['F_sha256']=array_digest(F)
    manifest['parent_feature_key']=data['feature_key']
    manifest['source_channels']=list(data['manifest']['channels'])
    manifest['channels']=[name for name in data['manifest']['channels'] if name in names]
    manifest['channel_count']=len(names)
    manifest['channel_configuration_id']=geometry['configuration_id']
    manifest['pooled_shape'][1]=len(names)
    manifest['feature_info']=dict(manifest['feature_info'],available_features=len(schema),features=len(schema))
    result=dict(data,F=F,schema=schema,manifest=manifest,
                feature_key=digest(dict(parent=data['feature_key'],channels=manifest['channels'],
                                       F_sha256=manifest['F_sha256'],geometry=geometry['configuration_id'])))
    if len(result['y'])!=len(result['F']) or not np.array_equal(result['y'],data['y']):
        raise ValueError('Channel masking changed trial/label alignment.')
    return result

def permuted_subject(data,permutation_id,config):
    if permutation_id<1:
        raise ValueError('Permutation IDs start at1; zero means the real-label experiment.')
    y=data['y'].copy()
    rng=np.random.default_rng(np.random.SeedSequence([config['seed'],int(data['subject']),int(permutation_id),104729]))
    for session,frame in data['metadata'].groupby('session',sort=True):
        rows=frame.index.to_numpy()
        y[rows]=rng.permutation(y[rows])
        assert np.array_equal(np.bincount(y[rows],minlength=4),np.bincount(data['y'][rows],minlength=4))
    metadata=data['metadata'].copy()
    metadata['original_class_id']=data['y']
    metadata['class_id']=y
    manifest=deepcopy(data['manifest'])
    manifest['permutation']=dict(permutation_id=permutation_id,within='subject/session',
                                target_sha256=array_digest(y),source_target_sha256=array_digest(data['y']))
    # Feature extraction and the channel set remain unchanged; label changes enter selection/cache identities.
    return dict(data,y=y,metadata=metadata,manifest=manifest,
                feature_key=digest(dict(parent=data['feature_key'],permutation=manifest['permutation'])))

def expected_roster(datasets,config):
    rows=[]
    if config['run_main']:
        for subject in datasets:
            for count in config['channel_counts']:
                for selector in config['selectors']:
                    for classifier in config['classifiers']:
                        for boundary in config['boundaries']:
                            rows.append(dict(subject=subject,channel_count=count,selector=selector,classifier=classifier,
                                             boundary=boundary,permutation_id=0,experiment='main'))
    if config['run_permutation_controls']:
        for subject in config['permutation_subjects']:
            for permutation_id in range(1,config['n_permutations']+1):
                for count in config['permutation_channel_counts']:
                    for selector in config['permutation_selectors']:
                        for classifier in config['permutation_classifiers']:
                            for boundary in config['boundaries']:
                                rows.append(dict(subject=subject,channel_count=count,selector=selector,classifier=classifier,
                                                 boundary=boundary,permutation_id=permutation_id,experiment='permutation'))
    return pd.DataFrame(rows,columns=['subject','channel_count','selector','classifier','boundary','permutation_id','experiment'])

def preflight_channel_inputs(datasets,channel_sets,geometry,config,root):
    rows=[]
    plans={}
    for subject,data in datasets.items():
        if not data['metadata'].index.equals(pd.RangeIndex(len(data['metadata']))):
            raise ValueError('Metadata must use the same positional row order as F/y.')
        plans[subject]=cv_plan(data,config)
        save_json(Path(root)/f'sub-{subject:02d}_real_label_plan.json',plans[subject])
        for count,names in channel_sets.items():
            view=channel_view(data,names,geometry)
            if config['k_features']>view['F'].shape[1]:
                raise ValueError(f'K={config["k_features"]} exceeds {view["F"].shape[1]} candidates at{count} channels. Lower K or change representation before training.')
            assert cv_plan(view,config)==plans[subject]
            rows.append(dict(subject=subject,channel_count=count,trials=len(view['y']),
                             available_features=view['F'].shape[1],selected_features=config['k_features'],
                             features_per_channel=view['F'].shape[1]//count,shape=str(view['F'].shape)))
    preview=pd.DataFrame(rows)
    atomic_csv(preview,Path(root)/'feature_dimensions.csv')
    roster=expected_roster(datasets,config)
    atomic_csv(roster,Path(root)/'requested_roster.csv')
    print(f'Planned completed-subject evaluations:{len(roster)}; outer classifier fits:{len(roster)*config["outer_folds"]} '
          '(inner tuning, when enabled, adds fits).')
    return preview,roster,plans

def evaluate_channel_task(view,selector,classifier,boundary,permutation_id,config,cache_root,active):
    count=view['manifest']['channel_count']
    spec=dict(stage='channel_reduction',selector=selector,k=config['k_features'],classifier=classifier,
              channel_count=count,permutation_id=permutation_id,
              channel_configuration_id=view['manifest']['channel_configuration_id'],
              candidate_features=view['F'].shape[1],label_condition='real' if permutation_id==0 else 'permuted')
    result=evaluate(view,spec,boundary,config,cache_root)
    original_indices=view['schema'].source_feature_index.to_numpy()
    result['selections']=result['selections'].copy()
    result['selections']['source_feature_index']=original_indices[result['selections'].feature_index.to_numpy()]
    used=result['selections'].groupby(['unit_session','fold']).channel.nunique()
    if not set(result['selections'].channel).issubset(view['manifest']['channels']) or used.max()>count:
        raise ValueError('Selected features use channels outside the fixed configuration.')
    result['row']['selected_channels_min']=int(used.min())
    result['row']['selected_channels_max']=int(used.max())
    result['row']['selected_channels_mean']=float(used.mean())
    result['row']['mean_train_accuracy']=float(result['fold_scores'].train_accuracy.mean())
    result['row']['elapsed_seconds_sum']=float(sum(a['elapsed_seconds'] for a in result['audits']))
    # Enrich resumed/recomputed outputs with the source feature mapping and fixed channel names.
    atomic_csv(result['selections'],result['directory']/'selected_features.csv')
    save_json(result['directory']/'channel_input.json',dict(subject=view['subject'],channel_count=count,
              channels=view['manifest']['channels'],configuration_id=spec['channel_configuration_id'],
              feature_key=view['feature_key'],F_sha256=view['manifest']['F_sha256'],
              parent_feature_key=view['manifest']['parent_feature_key'],permutation=view['manifest'].get('permutation'),
              trial_ids=view['metadata'].trial_id.tolist(),scored_target_sha256=array_digest(view['y'])))
    save_json(result['directory']/'result.json',result['row'])
    active[result['row']['result_id']]=result
    return result

def run_channel_stage(datasets,channel_sets,geometry,config,root,cache_root,active,status,experiment='main'):
    if experiment not in {'main','permutation'}:
        raise ValueError('experiment must be main or permutation.')
    if experiment=='main':
        subjects=list(datasets); ids=[0]; counts=config['channel_counts']
        selectors=config['selectors']; classifiers=config['classifiers']
    else:
        subjects=config['permutation_subjects']; ids=list(range(1,config['n_permutations']+1))
        counts=config['permutation_channel_counts']; selectors=config['permutation_selectors']; classifiers=config['permutation_classifiers']
    for subject in subjects:
        for permutation_id in ids:
            original=datasets[subject]
            data=original if permutation_id==0 else permuted_subject(original,permutation_id,config)
            plan=cv_plan(data,config)
            if permutation_id:
                save_json(Path(root)/f'sub-{subject:02d}_permutation-{permutation_id:03d}_plan.json',plan)
                target=data['metadata'][['trial_id','subject','session','original_class_id','class_id']].copy()
                target['permutation_id']=permutation_id
                atomic_csv(target,Path(root)/f'sub-{subject:02d}_permutation-{permutation_id:03d}_labels.csv')
            for count in counts:
                view=channel_view(data,channel_sets[count],geometry)
                assert cv_plan(view,config)==plan
                print(f'\n{experiment.upper()} | subject{subject:02d} | channels={count} | candidates={view["F"].shape[1]} | '
                      f'K={config["k_features"]} | permutation={permutation_id}',flush=True)
                for selector in selectors:
                    for classifier in classifiers:
                        if classifier=='lgbm' and importlib.util.find_spec('lightgbm') is None:
                            for boundary in config['boundaries']:
                                status.append(dict(subject=subject,channel_count=count,selector=selector,classifier=classifier,
                                                   boundary=boundary,permutation_id=permutation_id,experiment=experiment,
                                                   status='skipped',reason='LightGBM unavailable; no substitute result.'))
                            print('LightGBM unavailable: explicitly skipped.',flush=True)
                            continue
                        paired=[]
                        for boundary in config['boundaries']:
                            result=evaluate_channel_task(view,selector,classifier,boundary,permutation_id,config,cache_root,active)
                            paired.append(result['oof'])
                            atomic_csv(pd.DataFrame([r['row'] for r in active.values()]),Path(root)/'active_subject_results.csv')
                        assert paired[0].trial_id.tolist()==paired[1].trial_id.tolist()
                        assert np.array_equal(paired[0].y_true,paired[1].y_true)
                        assert np.array_equal(paired[0].outer_fold,paired[1].outer_fold)
    atomic_csv(pd.DataFrame(status,columns=['subject','channel_count','selector','classifier','boundary','permutation_id','experiment','status','reason']),
               Path(root)/'stage_status.csv')
    return pd.DataFrame([r['row'] for r in active.values() if (r['row']['permutation_id']==0)==(experiment=='main')])

def channel_aggregates(scores):
    columns=['feature_mode','selector','k','classifier','channel_count','boundary','selection_boundary','n_subjects',
             'accuracy_mean','accuracy_sd','balanced_accuracy_mean','balanced_accuracy_sd','macro_f1_mean','macro_f1_sd']
    if scores.empty:
        return pd.DataFrame(columns=columns)
    keys=columns[:7]
    rows=[]
    for setting,frame in scores.groupby(keys,dropna=False):
        if frame.subject.duplicated().any():
            raise ValueError('Subject aggregation received repeated subjects; do not treat permutations/folds as subjects.')
        row=dict(zip(keys,setting),n_subjects=frame.subject.nunique(),subjects=','.join(map(str,sorted(frame.subject))))
        for metric in ('accuracy','balanced_accuracy','macro_f1',*[f'recall_{i}' for i in LABELS]):
            row[metric+'_mean']=frame[metric].mean()
            row[metric+'_sd']=frame[metric].std(ddof=1)
        row['mean_train_accuracy']=frame.mean_train_accuracy.mean()
        row['mean_fit_seconds_per_subject']=frame.elapsed_seconds_sum.mean()
        rows.append(row)
    return pd.DataFrame(rows)

def paired_boundary_table(scores):
    keys=['subject','feature_mode','selector','k','classifier','channel_count','permutation_id']
    if scores.empty:
        return pd.DataFrame(columns=keys)
    a=scores[scores.boundary=='PAPER_ORDER']; b=scores[scores.boundary=='NESTED_SAFE']
    merged=a.merge(b,on=keys,suffixes=('_paper_order','_nested_safe'),validate='one_to_one')
    columns=keys.copy()
    for metric in ('accuracy','balanced_accuracy','macro_f1'):
        merged[metric+'_gap']=merged[metric+'_paper_order']-merged[metric+'_nested_safe']
        columns += [metric+'_paper_order',metric+'_nested_safe',metric+'_gap']
    merged['paper_order_label']=PAPER_LABEL
    return merged[columns+['paper_order_label']]

def changes_from_128(scores):
    keys=['subject','feature_mode','selector','k','classifier','boundary','permutation_id']
    if scores.empty:
        return pd.DataFrame(columns=keys)
    reference=scores[scores.channel_count==128]
    columns=keys+['accuracy','balanced_accuracy','macro_f1']
    merged=scores.merge(reference[columns],on=keys,suffixes=('','_128'),validate='many_to_one')
    for metric in ('accuracy','balanced_accuracy','macro_f1'):
        merged[metric+'_change_from_128']=merged[metric]-merged[metric+'_128']
    return merged[keys+['channel_count','selection_boundary']+
                  [name for metric in ('accuracy','balanced_accuracy','macro_f1') for name in (metric,metric+'_128',metric+'_change_from_128')]]

def export_channel_tables(active,status,roster,config,root):
    score_columns=['result_id','subject','feature_mode','scope','stage','selector','k','classifier','channel_count',
                   'permutation_id','boundary','selection_boundary','accuracy','balanced_accuracy','macro_f1',
                   'mean_train_accuracy','elapsed_seconds_sum']
    scores=pd.DataFrame([r['row'] for r in active.values()]) if active else pd.DataFrame(columns=score_columns)
    main=scores[scores.permutation_id==0].copy()
    controls=scores[scores.permutation_id>0].copy()
    summary=channel_aggregates(main)
    gaps=paired_boundary_table(main)
    changes=changes_from_128(main)
    check_keys=['subject','channel_count','selector','classifier','boundary','permutation_id']
    completed=scores[check_keys].copy()
    completed['status']='complete'
    roster_status=roster.merge(completed,on=check_keys,how='left',validate='one_to_one')
    if status:
        skipped=pd.DataFrame(status).drop_duplicates(check_keys,keep='last')
        roster_status=roster_status.merge(skipped[check_keys+['reason']],on=check_keys,how='left')
        roster_status.loc[roster_status.reason.notna(),'status']='skipped'
    roster_status['status']=roster_status.status.fillna('pending')
    if not summary.empty:
        summary['requested_subjects']=len(config['subjects'])
        summary['complete_subject_roster']=summary.n_subjects.eq(len(config['subjects']))
    by_permutation=[]
    for pid,frame in controls.groupby('permutation_id'):
        table=channel_aggregates(frame)
        table['permutation_id']=pid
        by_permutation.append(table)
    control_means=pd.concat(by_permutation,ignore_index=True) if by_permutation else pd.DataFrame()
    control_summary=[]
    control_keys=['feature_mode','selector','k','classifier','channel_count','boundary','selection_boundary']
    if not control_means.empty:
        for setting,frame in control_means.groupby(control_keys):
            control_summary.append(dict(zip(control_keys,setting),n_permutations=frame.permutation_id.nunique(),
                n_subjects=int(frame.n_subjects.min()),mean_balanced_accuracy=frame.balanced_accuracy_mean.mean(),
                sd_across_permutation_means=frame.balanced_accuracy_mean.std(ddof=1),
                min_balanced_accuracy=frame.balanced_accuracy_mean.min(),max_balanced_accuracy=frame.balanced_accuracy_mean.max()))
    control_summary=pd.DataFrame(control_summary)
    session_rows=[]
    for result in active.values():
        row=result['row']
        for session,group in result['oof'].groupby('session'):
            session_rows.append(dict(**{k:row[k] for k in ('result_id','subject','feature_mode','selector','k','classifier',
                  'channel_count','permutation_id','boundary','selection_boundary')},session=int(session),n_trials=len(group),
                  **metrics(group.y_true,group.y_pred)))
    sessions=pd.DataFrame(session_rows)
    for name,frame in [('per_subject_scores',main),('subject_mean_scores',summary),('paired_boundary_gaps',gaps),
                       ('changes_from_128',changes),('permutation_subject_scores',controls),
                       ('permutation_cohort_means',control_means),('permutation_summary',control_summary),
                       ('session_oof_scores',sessions),('completion_status',roster_status)]:
        atomic_csv(frame,Path(root)/(name+'.csv'))
    lines=['Scores are proportions. Means weight subjects equally. SD describes subjects, not independent CV folds.',
           'PAPER_ORDER uses held-out labels in selection and is an intentionally leaky diagnostic.',
           'NESTED_SAFE estimates held-out performance under the declared subject-specific pooled-session protocol.',
           'This is channel masking of processed high-density recordings, not a simulation of new low-density recordings.']
    for setting,frame in summary.groupby(['feature_mode','selector','classifier']) if not summary.empty else []:
        safe=frame[frame.boundary=='NESTED_SAFE'].sort_values('channel_count')
        lines.append(f'{setting}: training-only channel-count/BA means = '+str(dict(zip(safe.channel_count,safe.balanced_accuracy_mean.round(4)))))
        base=safe[safe.channel_count==128]
        if len(base) and abs(base.balanced_accuracy_mean.iloc[0]-.25)<=.05:
            lines.append(f'{setting}: 128-channel mean is near the25% reference; a flat reduction curve does not establish preservation of useful decoding.')
    if not control_summary.empty:
        lines.append('Randomized-label results are an initial diagnostic. A few permutations are not a formal significance test; permutation repetitions are not independent subjects.')
        for row in control_summary.itertuples(index=False):
            lines.append(f'Randomized labels, {row.classifier}/{row.selector}, {row.channel_count} channels, {row.selection_boundary}: '
                         f'mean BA={row.mean_balanced_accuracy:.1%} across{row.n_permutations} permutations and{row.n_subjects} subjects.')
    if roster_status.status.eq('pending').any():
        lines.append(f'{int(roster_status.status.eq("pending").sum())} requested evaluations remain pending. Partial results are labeled in completion_status.csv.')
    (Path(root)/'interpretation.txt').write_text('\n\n'.join(lines),encoding='utf-8')
    save_json(Path(root)/'run_manifest.json',dict(config=config,engine_id=ENGINE_ID,versions=environment_info(),
               source_inspection=SOURCE_INSPECTION,active_result_ids=list(active),expected_evaluations=len(roster),
               completed_evaluations=len(scores),skipped_evaluations=len(status),
               label_control_note='Labels randomized within subject/session, then selection, fitting and scoring use those labels. '
                                  'Each permutation is restratified once; its plan is shared across budgets and modes.'))
    for line in lines:
        print(line+'\n')
    return dict(subject_scores=main,summary=summary,gaps=gaps,changes=changes,
                controls=controls,control_summary=control_summary,sessions=sessions,completion=roster_status)

def plot_channel_performance(tables,root):
    means=tables['summary']
    if means.empty:
        print('No real-label results yet.')
        return
    for setting,frame in means.groupby(['feature_mode','selector','classifier']):
        fig,axes=plt.subplots(1,2,figsize=(13,4.8))
        colors={'PAPER_ORDER':'#d95f02','NESTED_SAFE':'#1b9e77'}
        for boundary,group in frame.groupby('boundary'):
            group=group.sort_values('channel_count')
            axes[0].errorbar(group.channel_count,100*group.balanced_accuracy_mean,
                            yerr=100*group.balanced_accuracy_sd.fillna(0),marker='o',capsize=3,
                            color=colors[boundary],label=boundary_label(boundary))
        mode,selector,classifier=setting
        gaps=tables['gaps']
        g=gaps[(gaps.feature_mode==mode)&(gaps.selector==selector)&(gaps.classifier==classifier)]
        gapmeans=g.groupby('channel_count').balanced_accuracy_gap.agg(['mean','std']).sort_index()
        axes[1].errorbar(gapmeans.index,100*gapmeans['mean'],yerr=100*gapmeans['std'].fillna(0),marker='o',capsize=3,color='#7570b3')
        axes[0].axhline(25,linestyle=':',color='#444444',label='Four-class chance reference')
        axes[0].set(ylabel='Balanced accuracy (%)',title='Subject mean ± SD (when at least two subjects)',ylim=(0,103))
        axes[1].set(ylabel='PAPER_ORDER − NESTED_SAFE (percentage points)',title='Paired subject boundary gap ± subject SD')
        axes[1].axhline(0,linestyle=':',color='#444444')
        for ax in axes:
            ax.set(xlabel='Available EEG channels',xticks=sorted(frame.channel_count.unique()))
            ax.grid(alpha=.2)
        axes[0].legend(fontsize=7,loc='best')
        fig.suptitle(f'{mode}, {selector.upper()}, {classifier.upper()} — fixed selected K, matched trials/folds; '
                     f'completed subjects per point: {int(frame.n_subjects.min())}–{int(frame.n_subjects.max())}')
        fig.tight_layout()
        fig.savefig(Path(root)/f'performance_{mode}_{selector}_{classifier}.png',dpi=150,bbox_inches='tight')
        plt.show()

def plot_changes_from_128(tables,root):
    changes=tables['changes']
    if changes.empty:
        return
    for setting,frame in changes.groupby(['feature_mode','selector','classifier']):
        fig,ax=plt.subplots(figsize=(8,4.8))
        for boundary,group in frame.groupby('boundary'):
            mean=group.groupby('channel_count').balanced_accuracy_change_from_128.mean().sort_index()
            ax.plot(mean.index,100*mean,marker='o',color=CHANNEL_BOUNDARY_COLORS[boundary],label=boundary_label(boundary))
        ax.axhline(0,color='#444444',linestyle=':')
        ax.set(xlabel='Available EEG channels',ylabel='BA change from 128 (percentage points)',
               xticks=sorted(frame.channel_count.unique()),title=f'{" | ".join(setting)}: paired changes from each subject’s 128-channel score')
        ax.grid(alpha=.2); ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(Path(root)/('change_from128_'+'_'.join(setting)+'.png'),dpi=150,bbox_inches='tight')
        plt.show()

def plot_channel_subjects(tables,root):
    scores=tables['subject_scores']
    if scores.empty:
        return
    for setting,frame in scores.groupby(['feature_mode','selector','classifier']):
        subjects=sorted(frame.subject.unique())
        ncols=min(3,len(subjects)); nrows=int(np.ceil(len(subjects)/ncols))
        fig,axes=plt.subplots(nrows,ncols,figsize=(5*ncols,3.6*nrows),squeeze=False)
        for ax,subject in zip(axes.flat,subjects):
            for boundary,group in frame[frame.subject==subject].groupby('boundary'):
                group=group.sort_values('channel_count')
                ax.plot(group.channel_count,100*group.balanced_accuracy,marker='o',color=CHANNEL_BOUNDARY_COLORS[boundary],label=boundary_label(boundary))
            ax.axhline(25,color='#444444',linestyle=':')
            ax.set(title=f'Subject {subject:02d}',xlabel='Channels',ylabel='BA (%)',ylim=(0,103),xticks=sorted(frame.channel_count.unique()))
            ax.grid(alpha=.2)
        for ax in list(axes.flat)[len(subjects):]:
            ax.axis('off')
        axes.flat[0].legend(fontsize=6)
        fig.suptitle(f'{" | ".join(setting)}: subject-specific channel-reduction curves')
        fig.tight_layout()
        fig.savefig(Path(root)/('subjects_'+'_'.join(setting)+'.png'),dpi=140,bbox_inches='tight')
        plt.show()

def plot_permutation_controls(tables,root):
    controls=tables['control_summary']
    if controls.empty:
        print('Permutation controls disabled or not completed.')
        return
    for setting,frame in controls.groupby(['feature_mode','selector','classifier']):
        fig,ax=plt.subplots(figsize=(9,4.8))
        for boundary,group in frame.groupby('boundary'):
            group=group.sort_values('channel_count')
            ax.plot(group.channel_count,100*group.mean_balanced_accuracy,marker='o',color=CHANNEL_BOUNDARY_COLORS[boundary],label=boundary_label(boundary))
            ax.fill_between(group.channel_count,100*group.min_balanced_accuracy,100*group.max_balanced_accuracy,color=CHANNEL_BOUNDARY_COLORS[boundary],alpha=.12)
        ax.axhline(25,linestyle=':',color='#444444')
        ax.set(xlabel='EEG channels',ylabel='BA against randomized labels (%)',ylim=(0,103),
               xticks=sorted(frame.channel_count.unique()),title=f'{" | ".join(setting)}: shuffled labels\nShading: min–max permutation means; not a confidence interval')
        ax.grid(alpha=.2); ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(Path(root)/('randomized_labels_'+'_'.join(setting)+'.png'),dpi=150,bbox_inches='tight')
        plt.show()

def export_channel_review(active,config,root):
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    target=Path(config['output_root'])/f'channel_reduction_review_{stamp}.zip'
    root=Path(root)
    # Reports belong to the current config. Include only currently active cached runs.
    with ZipFile(target,'w',ZIP_DEFLATED) as archive:
        for path in sorted(root.iterdir()):
            if path.is_file() and path.suffix in {'.csv','.json','.txt','.png','.py'}:
                archive.write(path,path.name)
        for result in active.values():
            for path in sorted(result['directory'].iterdir()):
                if path.is_file() and path.suffix in {'.csv','.json'}:
                    archive.write(path,f'runs/{result["row"]["result_id"]}/{path.name}')
    print(f'Review ZIP:{target} ({target.stat().st_size/1024**2:.2f} MB). No raw EEG or feature arrays included.')
    return target


## 4. Embedded official CBraMod architecture and provenance

The architecture below is the authors' MIT-licensed implementation pinned to source commit `b9e961003214326972c567eff390e75b0287e32a`. Only package-local imports/script entry points and the functional alias are adjusted. All 211 checkpoint tensors must match strictly before the reconstruction head is replaced.

CBraMod receives `(batch, retained_channels, time_patches, 200)`. Input scaling is fixed microvolts/100, following the authors' downstream inner-speech loader. Time-patch representations are averaged and **channel identities are retained by flattening in source order**. A classifier is trained separately for each subject and channel budget. Sparse channel subsets change the spatial neighbourhoods seen by the encoder; this is a transfer-adaptation assumption, not a claim of exact anatomical alignment with pretraining.

Frozen embeddings may be computed for all trials before classifier CV because the encoder, input transform and pooling are fixed, label-free and trial-local. The downstream scaler/classifier still fit training rows only. Fine-tuning instead starts from the original checkpoint independently for every inner phase and outer refit. No weights are carried from one fold or subject to the next.


In [ ]:
PRETRAINED_PROVENANCE = {'source_commit': 'b9e961003214326972c567eff390e75b0287e32a', 'checkpoint_revision': '500543c7e30bda1b22bfd51a49301b238dee21fd', 'checkpoint_repo': 'weighting666/CBraMod', 'checkpoint_filename': 'pretrained_weights.pth', 'checkpoint_sha256': '0792cb808c14e6b7a2bb2ce1dff379bc47bc54c49a779825bdfeb33bf8157178', 'source_url': 'https://github.com/wjq-learning/CBraMod', 'adaptations': ['Embed model definitions; remove package-local import and script entry points.', 'Rename functional alias to torch_F to avoid conflict with user feature variable F.', 'Strict-load all pretrained parameters; then remove reconstruction head.', 'Mean-pool time patches; flatten channels in canonical source order for a downstream classifier.']}
UPSTREAM_LICENSE = 'MIT License\n\nCopyright (c) 2025 Jiquan Wang\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'
import copy

from typing import Optional, Any, Union, Callable

import torch

import torch.nn as nn

import warnings

from torch import Tensor

from torch.nn import functional as torch_F

class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None, enable_nested_tensor=True, mask_check=True):
        super().__init__()
        torch._C._log_api_usage_once(f"torch.nn.modules.{self.__class__.__name__}")
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(
            self,
            src: Tensor,
            mask: Optional[Tensor] = None,
            src_key_padding_mask: Optional[Tensor] = None,
            is_causal: Optional[bool] = None) -> Tensor:

        output = src
        for mod in self.layers:
            output = mod(output, src_mask=mask)
        if self.norm is not None:
            output = self.norm(output)
        return output

class TransformerEncoderLayer(nn.Module):
    __constants__ = ['norm_first']

    def __init__(self, d_model: int, nhead: int, dim_feedforward: int = 2048, dropout: float = 0.1,
                 activation: Union[str, Callable[[Tensor], Tensor]] = torch_F.relu,
                 layer_norm_eps: float = 1e-5, batch_first: bool = False, norm_first: bool = False,
                 bias: bool = True, device=None, dtype=None) -> None:
        factory_kwargs = {'device': device, 'dtype': dtype}
        super().__init__()
        self.self_attn_s = nn.MultiheadAttention(d_model//2, nhead // 2, dropout=dropout,
                                                 bias=bias, batch_first=batch_first,
                                                 **factory_kwargs)
        self.self_attn_t = nn.MultiheadAttention(d_model//2, nhead // 2, dropout=dropout,
                                                 bias=bias, batch_first=batch_first,
                                                 **factory_kwargs)

        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward, bias=bias, **factory_kwargs)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model, bias=bias, **factory_kwargs)

        self.norm_first = norm_first
        self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps, **factory_kwargs)
        self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps, **factory_kwargs)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        # Legacy string support for activation function.
        if isinstance(activation, str):
            activation = _get_activation_fn(activation)

        # We can't test self.activation in forward() in TorchScript,
        # so stash some information about it instead.
        if activation is torch_F.relu or isinstance(activation, torch.nn.ReLU):
            self.activation_relu_or_gelu = 1
        elif activation is torch_F.gelu or isinstance(activation, torch.nn.GELU):
            self.activation_relu_or_gelu = 2
        else:
            self.activation_relu_or_gelu = 0
        self.activation = activation

    def __setstate__(self, state):
        super().__setstate__(state)
        if not hasattr(self, 'activation'):
            self.activation = torch_F.relu


    def forward(
            self,
            src: Tensor,
            src_mask: Optional[Tensor] = None,
            src_key_padding_mask: Optional[Tensor] = None,
            is_causal: bool = False) -> Tensor:

        x = src
        x = x + self._sa_block(self.norm1(x), src_mask, src_key_padding_mask, is_causal=is_causal)
        x = x + self._ff_block(self.norm2(x))
        return x

    # self-attention block
    def _sa_block(self, x: Tensor,
                  attn_mask: Optional[Tensor], key_padding_mask: Optional[Tensor], is_causal: bool = False) -> Tensor:
        bz, ch_num, patch_num, patch_size = x.shape
        xs = x[:, :, :, :patch_size // 2]
        xt = x[:, :, :, patch_size // 2:]
        xs = xs.transpose(1, 2).contiguous().view(bz*patch_num, ch_num, patch_size // 2)
        xt = xt.contiguous().view(bz*ch_num, patch_num, patch_size // 2)
        xs = self.self_attn_s(xs, xs, xs,
                             attn_mask=attn_mask,
                             key_padding_mask=key_padding_mask,
                             need_weights=False)[0]
        xs = xs.contiguous().view(bz, patch_num, ch_num, patch_size//2).transpose(1, 2)
        xt = self.self_attn_t(xt, xt, xt,
                              attn_mask=attn_mask,
                              key_padding_mask=key_padding_mask,
                              need_weights=False)[0]
        xt = xt.contiguous().view(bz, ch_num, patch_num, patch_size//2)
        x = torch.concat((xs, xt), dim=3)
        return self.dropout1(x)

    # feed forward block
    def _ff_block(self, x: Tensor) -> Tensor:
        x = self.linear2(self.dropout(self.activation(self.linear1(x))))
        return self.dropout2(x)

def _get_activation_fn(activation: str) -> Callable[[Tensor], Tensor]:
    if activation == "relu":
        return torch_F.relu
    elif activation == "gelu":
        return torch_F.gelu

    raise RuntimeError(f"activation should be relu/gelu, not {activation}")

def _get_clones(module, N):
    # FIXME: copy.deepcopy() is not defined on nn.module
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

def _get_seq_len(
        src: Tensor,
        batch_first: bool
) -> Optional[int]:

    if src.is_nested:
        return None
    else:
        src_size = src.size()
        if len(src_size) == 2:
            # unbatched: S, E
            return src_size[0]
        else:
            # batched: B, S, E if batch_first else S, B, E
            seq_len_pos = 1 if batch_first else 0
            return src_size[seq_len_pos]

def _detect_is_causal_mask(
        mask: Optional[Tensor],
        is_causal: Optional[bool] = None,
        size: Optional[int] = None,
) -> bool:
    """Return whether the given attention mask is causal.

    Warning:
    If ``is_causal`` is not ``None``, its value will be returned as is.  If a
    user supplies an incorrect ``is_causal`` hint,

    ``is_causal=False`` when the mask is in fact a causal attention.mask
       may lead to reduced performance relative to what would be achievable
       with ``is_causal=True``;
    ``is_causal=True`` when the mask is in fact not a causal attention.mask
       may lead to incorrect and unpredictable execution - in some scenarios,
       a causal mask may be applied based on the hint, in other execution
       scenarios the specified mask may be used.  The choice may not appear
       to be deterministic, in that a number of factors like alignment,
       hardware SKU, etc influence the decision whether to use a mask or
       rely on the hint.
    ``size`` if not None, check whether the mask is a causal mask of the provided size
       Otherwise, checks for any causal mask.
    """
    # Prevent type refinement
    make_causal = (is_causal is True)

    if is_causal is None and mask is not None:
        sz = size if size is not None else mask.size(-2)
        causal_comparison = _generate_square_subsequent_mask(
            sz, device=mask.device, dtype=mask.dtype)

        # Do not use `torch.equal` so we handle batched masks by
        # broadcasting the comparison.
        if mask.size() == causal_comparison.size():
            make_causal = bool((mask == causal_comparison).all())
        else:
            make_causal = False

    return make_causal

def _generate_square_subsequent_mask(
        sz: int,
        device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),
        dtype: torch.dtype = torch.get_default_dtype(),
) -> Tensor:
    r"""Generate a square causal mask for the sequence. The masked positions are filled with float('-inf').
        Unmasked positions are filled with float(0.0).
    """
    return torch.triu(
        torch.full((sz, sz), float('-inf'), dtype=dtype, device=device),
        diagonal=1,
    )

import torch

import torch.nn as nn

import torch.nn.functional as torch_F

class CBraMod(nn.Module):
    def __init__(self, in_dim=200, out_dim=200, d_model=200, dim_feedforward=800, seq_len=30, n_layer=12,
                    nhead=8):
        super().__init__()
        self.patch_embedding = PatchEmbedding(in_dim, out_dim, d_model, seq_len)
        encoder_layer = TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, batch_first=True, norm_first=True,
            activation=torch_F.gelu
        )
        self.encoder = TransformerEncoder(encoder_layer, num_layers=n_layer, enable_nested_tensor=False)
        self.proj_out = nn.Sequential(
            # nn.Linear(d_model, d_model*2),
            # nn.GELU(),
            # nn.Linear(d_model*2, d_model),
            # nn.GELU(),
            nn.Linear(d_model, out_dim),
        )
        self.apply(_weights_init)

    def forward(self, x, mask=None):
        patch_emb = self.patch_embedding(x, mask)
        feats = self.encoder(patch_emb)

        out = self.proj_out(feats)

        return out

class PatchEmbedding(nn.Module):
    def __init__(self, in_dim, out_dim, d_model, seq_len):
        super().__init__()
        self.d_model = d_model
        self.positional_encoding = nn.Sequential(
            nn.Conv2d(in_channels=d_model, out_channels=d_model, kernel_size=(19, 7), stride=(1, 1), padding=(9, 3),
                      groups=d_model),
        )
        self.mask_encoding = nn.Parameter(torch.zeros(in_dim), requires_grad=False)
        # self.mask_encoding = nn.Parameter(torch.randn(in_dim), requires_grad=True)

        self.proj_in = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=25, kernel_size=(1, 49), stride=(1, 25), padding=(0, 24)),
            nn.GroupNorm(5, 25),
            nn.GELU(),

            nn.Conv2d(in_channels=25, out_channels=25, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1)),
            nn.GroupNorm(5, 25),
            nn.GELU(),

            nn.Conv2d(in_channels=25, out_channels=25, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1)),
            nn.GroupNorm(5, 25),
            nn.GELU(),
        )
        self.spectral_proj = nn.Sequential(
            nn.Linear(101, d_model),
            nn.Dropout(0.1),
            # nn.LayerNorm(d_model, eps=1e-5),
        )
        # self.norm1 = nn.LayerNorm(d_model, eps=1e-5)
        # self.norm2 = nn.LayerNorm(d_model, eps=1e-5)
        # self.proj_in = nn.Sequential(
        #     nn.Linear(in_dim, d_model, bias=False),
        # )


    def forward(self, x, mask=None):
        bz, ch_num, patch_num, patch_size = x.shape
        if mask == None:
            mask_x = x
        else:
            mask_x = x.clone()
            mask_x[mask == 1] = self.mask_encoding

        mask_x = mask_x.contiguous().view(bz, 1, ch_num * patch_num, patch_size)
        patch_emb = self.proj_in(mask_x)
        patch_emb = patch_emb.permute(0, 2, 1, 3).contiguous().view(bz, ch_num, patch_num, self.d_model)

        mask_x = mask_x.contiguous().view(bz*ch_num*patch_num, patch_size)
        spectral = torch.fft.rfft(mask_x, dim=-1, norm='forward')
        spectral = torch.abs(spectral).contiguous().view(bz, ch_num, patch_num, 101)
        spectral_emb = self.spectral_proj(spectral)
        # print(patch_emb[5, 5, 5, :])
        # print(spectral_emb[5, 5, 5, :])
        patch_emb = patch_emb + spectral_emb

        positional_embedding = self.positional_encoding(patch_emb.permute(0, 3, 1, 2))
        positional_embedding = positional_embedding.permute(0, 2, 3, 1)

        patch_emb = patch_emb + positional_embedding

        return patch_emb

def _weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    if isinstance(m, nn.Conv1d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    elif isinstance(m, nn.BatchNorm1d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)


## 5. Neural training, shared-fold audits and result exports

Scratch CNN normalization is fitted on inner training rows during epoch selection and on all outer training rows during the final refit. CBraMod uses its fixed input scaling; frozen-probe embedding standardization fits outer training rows only. No outer test labels control stopping, gradients, scaling or checkpoint choice.

The same original trial folds are saved for every model family. Each original trial receives exactly one OOF prediction. Within-session evaluation uses a distinct fold model for each session; its pooled subject OOF score summarizes all those within-session predictions.


In [ ]:
"""Shared EEG inputs, scratch CNN, pretrained transfer, and paired reports."""
import copy
import random
import urllib.request
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

THREE_PIPELINE_VERSION = 'three-pipelines-1.0'
_BASE_CV_PLAN = cv_plan
_BASE_VALIDATE_CHANNEL_CONFIG = validate_channel_config
_BASE_EVALUATE = evaluate

def cv_plan(data, config, scope=None):
    return _BASE_CV_PLAN(data, config, scope or config['evaluation_scope'])

def evaluate(data, spec, boundary, config, root, scope=None):
    return _BASE_EVALUATE(data, spec, boundary, config, root, scope or config['evaluation_scope'])

def validate_channel_config(config):
    temporary = copy.deepcopy(config)
    temporary['outer_folds'] = 10
    _BASE_VALIDATE_CHANNEL_CONFIG(temporary)
    if not isinstance(config['outer_folds'], int) or config['outer_folds'] < 2:
        raise ValueError('outer_folds must be an integer >=2.')
    if config['evaluation_scope'] not in {'pooled_subject', 'within_session'}:
        raise ValueError('evaluation_scope must be pooled_subject or within_session.')
    if not any(config[k] for k in ('run_ml', 'run_dl', 'run_pretrained')):
        raise ValueError('Enable at least one pipeline.')
    d, p = config['dl'], config['pretrained']
    if not p['modes'] or not set(p['modes']).issubset({'frozen_probe', 'finetune'}):
        raise ValueError('pretrained.modes: frozen_probe and/or finetune.')
    if len(p['modes']) != len(set(p['modes'])):
        raise ValueError('Duplicate pretrained modes.')
    if p['checkpoint_sha256'] != PRETRAINED_PROVENANCE['checkpoint_sha256']:
        raise ValueError('This notebook verifies the pinned official checkpoint; changing models requires a new adapter.')
    if config['neural_sfreq'] != 200. or config['neural_bandpass'][0] <= 0 or config['neural_bandpass'][1] >= 100:
        raise ValueError('CBraMod requires 200 Hz patches; use a valid bandpass below 100 Hz.')
    for parameters in (d, p):
        for key in ('max_epochs', 'patience', 'batch_size', 'print_every'):
            if not isinstance(parameters[key], int) or parameters[key] < 1:
                raise ValueError(f'{key} must be a positive integer.')
        if parameters['lr'] <= 0 or parameters['weight_decay'] < 0:
            raise ValueError('Use positive lr and nonnegative weight_decay.')
    if p['embedding_batch_size'] < 1 or p['probe_c'] <= 0:
        raise ValueError('Invalid embedding batch size or probe C.')
    if config['torch_threads'] < 1:
        raise ValueError('torch_threads must be positive.')
    if config['device'] not in {'auto', 'cpu', 'cuda'}:
        raise ValueError('device must be auto/cpu/cuda.')
    if config['device'] == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable. Enable a Kaggle GPU or use device=auto.')
    return config

def neural_device(config):
    return torch.device('cuda' if config['device'] == 'auto' and torch.cuda.is_available()
                        else 'cpu' if config['device'] == 'auto' else config['device'])

def seed_torch(seed, config):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(config['torch_threads'])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Warn rather than failing on an unsupported deterministic CUDA operation.
    torch.use_deterministic_algorithms(True, warn_only=True)

def attach_neural_input(data, config):
    """Read the same trusted FIF trials as the audited feature loader and verify IDs."""
    import mne
    files = [Path(p['path']) for p in data['manifest']['sources'] if str(p['path']).endswith('_eeg-epo.fif')]
    arrays, ids, labels = [], [], []
    source_names = data['manifest']['channels']
    for path in files:
        session = int(re.search(r'_ses-(\d+)_', path.name)[1])
        events = np.asarray(pd.read_pickle(path.with_name(path.name.replace('_eeg-epo.fif', '_events.dat')))).astype(np.int64)
        epochs = mne.read_epochs(path, preload=False, proj=False, verbose='ERROR')
        if events.shape != (len(epochs), 4) or not np.array_equal(events[:, 0], epochs.events[:, 0]) or not np.array_equal(events[:, 1]+31, epochs.events[:, 2]):
            raise ValueError('Raw EEG/event alignment failed.')
        selected = np.flatnonzero(events[:, 2] == 1)
        original = np.asarray(epochs.selection)[selected]
        epochs = epochs[selected].load_data()
        epochs.pick(mne.pick_types(epochs.info, eeg=True, exclude=[]))
        if epochs.ch_names != source_names or float(epochs.info['sfreq']) != data['manifest']['sfreq']:
            raise ValueError('Raw/feature channel order or sample rate differs.')
        start, stop = config['action_window_mne']
        mask = (epochs.times >= start) & (epochs.times < stop)
        arrays.append(epochs.get_data(copy=False)[:, :, mask].astype(np.float32))
        ids.extend(f"sub-{data['subject']:02d}_ses-{session:02d}_row-{i:04d}" for i in original)
        labels.extend(events[selected, 1])
    X = np.concatenate(arrays)
    if ids != data['metadata'].trial_id.tolist() or not np.array_equal(labels, data['y']):
        raise ValueError('ML and neural inputs do not contain the same ordered original trials.')
    if list(X.shape) != data['manifest']['pooled_shape'] or not np.isfinite(X).all():
        raise ValueError('Invalid raw input shape/values.')
    # Every operation is per trial/channel, with no label access or fitted cohort statistics.
    sfreq = data['manifest']['sfreq']
    sos = signal.butter(4, config['neural_bandpass'], btype='bandpass', fs=sfreq, output='sos')
    filtered = signal.sosfiltfilt(sos, X.astype(np.float64)*1e6, axis=-1)
    ratio = Fraction(config['neural_sfreq']/sfreq).limit_denominator(10000)
    X_neural = signal.resample_poly(filtered, ratio.numerator, ratio.denominator, axis=-1).astype(np.float32)
    payload = dict(shape=list(X_neural.shape), sfreq=config['neural_sfreq'], units='microvolts',
                   bandpass=config['neural_bandpass'], transform='Per-trial/channel SOS filtering and polyphase resampling.',
                   X_sha256=array_digest(X_neural), trial_ids=ids, source_feature_key=data['feature_key'])
    data = dict(data, X=X, X_neural=X_neural, neural_manifest=payload, neural_key=digest(payload))
    print(f"Subject {data['subject']:02d}: neural EEG {X_neural.shape}, 200 Hz, microvolts; verified against ML rows.", flush=True)
    return data

class ChannelStandardizer:
    """Fit one mean/std per retained channel using training trials/time points only."""
    def fit(self, X):
        self.mean_ = X.mean(axis=(0, 2), keepdims=True, dtype=np.float64)
        self.scale_ = np.maximum(X.std(axis=(0, 2), keepdims=True, dtype=np.float64), 1e-6)
        return self
    def transform(self, X):
        return ((X-self.mean_)/self.scale_).astype(np.float32)

class ScratchCNN(nn.Module):
    """Paper-inspired two Conv1d blocks; dimensions/head are declared adaptations."""
    def __init__(self, n_chans, config):
        super().__init__()
        a, b = config['widths']
        self.net = nn.Sequential(
            nn.Conv1d(n_chans, a, kernel_size=3, padding=1), nn.ELU(), nn.MaxPool1d(4), nn.Dropout(.1),
            nn.Conv1d(a, b, kernel_size=3, padding=1), nn.ELU(), nn.MaxPool1d(4), nn.Dropout(.1),
            nn.AdaptiveAvgPool1d(4), nn.Flatten(), nn.Linear(4*b, config['head_width']),
            nn.ELU(), nn.Dropout(.5), nn.Linear(config['head_width'], 4))
    def forward(self, x):
        return self.net(x)

def pretrained_checkpoint(config):
    p = config['pretrained']
    path = Path(p['checkpoint_path']) if p['checkpoint_path'] else Path(config['output_root'])/'pretrained'/p['checkpoint_filename']
    if not path.is_file():
        if not p['allow_download']:
            raise FileNotFoundError('Set pretrained.checkpoint_path to the official pretrained_weights.pth uploaded as Kaggle input, or enable allow_download and Kaggle Internet.')
        path.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://huggingface.co/{p['checkpoint_repo']}/resolve/{p['checkpoint_revision']}/{p['checkpoint_filename']}"
        temporary = path.with_suffix('.download')
        print('Downloading pinned official CBraMod checkpoint (~20 MB)...', flush=True)
        try:
            urllib.request.urlretrieve(url, temporary)
        except Exception as exc:
            raise RuntimeError('Checkpoint download failed. Enable Kaggle Internet or upload the official checkpoint and set checkpoint_path.') from exc
        if file_sha256(temporary) != p['checkpoint_sha256']:
            raise ValueError('Downloaded pretrained checkpoint hash mismatch.')
        temporary.replace(path)
    if file_sha256(path) != p['checkpoint_sha256']:
        raise ValueError('Pretrained checkpoint is not the verified pinned official file.')
    return path

def load_frozen_encoder(config):
    checkpoint = pretrained_checkpoint(config)
    # Restricted tensor-only loading; strict matching prevents a random-weight fallback.
    state = torch.load(checkpoint, map_location='cpu', weights_only=True)
    model = CBraMod()
    model.load_state_dict(state, strict=True)
    model.proj_out = nn.Identity()
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model

def cbramod_patches(X):
    """Microvolt inputs -> /100 scaling -> 200-sample patches, retaining the full interval."""
    n, c, t = X.shape
    pad = (-t) % 200
    # Padding occurs after division, so fixed zero padding represents zero EEG amplitude.
    values = np.pad(X/100., ((0, 0), (0, 0), (0, pad)), mode='constant')
    return values.reshape(n, c, -1, 200).astype(np.float32)

class PretrainedClassifier(nn.Module):
    def __init__(self, frozen_encoder, n_chans, config):
        super().__init__()
        self.encoder = copy.deepcopy(frozen_encoder)
        for p in self.encoder.parameters():
            p.requires_grad_(True)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(n_chans*200, config['head_width']),
                                  nn.ELU(), nn.Dropout(config['dropout']), nn.Linear(config['head_width'], 4))
    def forward(self, x):
        # Pool time patches but preserve the identity/order of every available channel.
        return self.head(self.encoder(x).mean(dim=2))

def frozen_embeddings(data, channel_names, encoder, config, cache_root):
    indices = [data['manifest']['channels'].index(name) for name in channel_names]
    X = data['X_neural'][:, indices]
    key = digest(dict(neural_key=data['neural_key'], channels=channel_names,
                      checkpoint=config['pretrained']['checkpoint_sha256'], engine=ENGINE_ID,
                      pooling='mean time patches; flatten channel x embedding', torch=torch.__version__))
    directory = Path(cache_root)/'embeddings'/key
    path = directory/'embeddings.npz'
    if config['resume'] and path.exists():
        with np.load(path) as z:
            E = z['E']
        if E.shape != (len(data['y']), len(indices)*200) or not np.isfinite(E).all():
            raise ValueError('Invalid pretrained embedding cache.')
        print(f"  Reused frozen embeddings {E.shape}.", flush=True)
        return E, key
    device = neural_device(config)
    encoder.to(device).eval()
    patches = cbramod_patches(X)
    parts = []
    batch_size = config['pretrained']['embedding_batch_size']
    with torch.inference_mode():
        for start in range(0, len(X), batch_size):
            batch = torch.from_numpy(patches[start:start+batch_size]).to(device)
            out = encoder(batch).mean(dim=2).flatten(1)
            parts.append(out.cpu().numpy())
            if start == 0 or (start//batch_size+1) % 10 == 0 or start+batch_size >= len(X):
                print(f'  Frozen CBraMod: {min(start+batch_size,len(X))}/{len(X)} original trials.', flush=True)
    encoder.to('cpu')
    E = np.concatenate(parts).astype(np.float32)
    if E.shape != (len(X), len(indices)*200) or not np.isfinite(E).all():
        raise ValueError('Invalid pretrained representations.')
    directory.mkdir(parents=True, exist_ok=True)
    atomic_npz(path, E=E)
    save_json(directory/'manifest.json', dict(key=key, channels=channel_names, checkpoint=PRETRAINED_PROVENANCE,
              trial_ids=data['metadata'].trial_id.tolist(), shape=list(E.shape), sha256=array_digest(E),
              label_access=False, encoder_training=False, normalization='Fixed microvolt /100; zero-pad final partial patch.'))
    return E, key

def neural_probabilities(model, X, device, batch_size):
    model.eval()
    parts = []
    with torch.inference_mode():
        for start in range(0, len(X), batch_size):
            logits = model(torch.from_numpy(X[start:start+batch_size]).to(device))
            parts.append(logits.softmax(dim=1).cpu().numpy())
    return np.concatenate(parts)

def train_neural(model, X, y, parameters, config, seed, X_valid=None, y_valid=None, fixed_epochs=None):
    """Validation chooses duration; the final refit receives no validation/test data."""
    seed_torch(seed, config)
    device = neural_device(config)
    model.to(device)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y.astype(np.int64))),
                        batch_size=parameters['batch_size'], shuffle=True, generator=generator, num_workers=0)
    if isinstance(model, PretrainedClassifier):
        optimizer = torch.optim.AdamW([
            dict(params=model.encoder.parameters(), lr=parameters['encoder_lr']),
            dict(params=model.head.parameters(), lr=parameters['lr'])], weight_decay=parameters['weight_decay'])
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=parameters['lr'], weight_decay=parameters['weight_decay'])
    criterion = nn.CrossEntropyLoss()
    best_loss, best_epoch, stale, history = np.inf, 1, 0, []
    epochs = fixed_epochs or parameters['max_epochs']
    for epoch in range(1, epochs+1):
        model.train()
        total_loss, correct, total = 0., 0, 0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(bx)
            loss = criterion(logits, by)
            if not torch.isfinite(loss):
                raise ValueError('Nonfinite neural loss; inspect input scaling.')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), parameters['gradient_clip'])
            optimizer.step()
            total_loss += loss.item()*len(by)
            correct += (logits.argmax(dim=1)==by).sum().item()
            total += len(by)
        row = dict(epoch=epoch, train_loss=total_loss/total, train_accuracy=correct/total)
        if X_valid is not None:
            prob = neural_probabilities(model, X_valid, device, parameters['batch_size'])
            valid_loss = float(-np.log(np.maximum(prob[np.arange(len(y_valid)), y_valid], 1e-12)).mean())
            row.update(validation_loss=valid_loss, validation_balanced_accuracy=balanced_accuracy_score(y_valid, prob.argmax(axis=1)))
            if valid_loss < best_loss-1e-6:
                best_loss, best_epoch, stale = valid_loss, epoch, 0
            else:
                stale += 1
        history.append(row)
        if epoch==1 or epoch%parameters['print_every']==0 or epoch==epochs:
            print(f"    Epoch {epoch}/{epochs}: train loss={row['train_loss']:.4f}"+
                  (f", inner loss={row['validation_loss']:.4f}" if X_valid is not None else ''), flush=True)
        if X_valid is not None and stale >= parameters['patience']:
            break
    return model, (best_epoch if X_valid is not None else epochs), history

def fit_neural_outer(X, y, train, fold_seed, kind, config, encoder):
    parameters = config['dl'] if kind=='scratch_cnn' else config['pretrained']
    counts = np.bincount(y[train], minlength=4)
    if counts.min() < config['inner_folds']:
        raise ValueError('Not enough outer-training trials for the configured inner validation folds.')
    # A single, predeclared inner split selects epoch count only; no model/HP grid is searched.
    splitter = StratifiedKFold(config['inner_folds'], shuffle=True, random_state=fold_seed)
    a, b = next(splitter.split(np.zeros(len(train)), y[train]))
    inner_train, inner_valid = train[a], train[b]
    if kind=='scratch_cnn':
        inner_scaler = ChannelStandardizer().fit(X[inner_train])
        A, B = inner_scaler.transform(X[inner_train]), inner_scaler.transform(X[inner_valid])
        final_scaler = ChannelStandardizer().fit(X[train])
        final_X = final_scaler.transform(X)
        make = lambda: ScratchCNN(X.shape[1], parameters)
    else:
        A, B = cbramod_patches(X[inner_train]), cbramod_patches(X[inner_valid])
        final_X = cbramod_patches(X)
        make = lambda: PretrainedClassifier(encoder, X.shape[1], parameters)
    seed_torch(fold_seed, config)
    search_model, chosen_epoch, search_history = train_neural(make(), A, y[inner_train], parameters, config, fold_seed,
                                                            X_valid=B, y_valid=y[inner_valid])
    del search_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    seed_torch(fold_seed, config)
    final_model, _, refit_history = train_neural(make(), final_X[train], y[train], parameters, config,
                                              fold_seed, fixed_epochs=chosen_epoch)
    audit = dict(inner_train_rows=inner_train.tolist(), inner_validation_rows=inner_valid.tolist(),
                 inner_scaler_fit_rows=inner_train.tolist() if kind=='scratch_cnn' else [],
                 scaler_fit_rows=train.tolist() if kind=='scratch_cnn' else [],
                 model_fit_rows=train.tolist(), epoch_selection_rows=inner_valid.tolist(),
                 chosen_epoch=chosen_epoch, stopping_metric='inner validation cross-entropy',
                 early_stopping='Select epoch inside outer train, reinitialize, refit all outer train for chosen duration.',
                 search_history=search_history, refit_history=refit_history)
    return final_model, final_X, audit

def evaluate_neural_task(data, channel_names, kind, config, cache_root, active, encoder=None):
    plan = cv_plan(data, config)
    indices = [data['manifest']['channels'].index(name) for name in channel_names]
    X = np.ascontiguousarray(data['X_neural'][:, indices])
    setting = dict(subject=data['subject'], channel_count=len(indices), scope=config['evaluation_scope'],
                   pipeline='DL' if kind=='scratch_cnn' else 'PRETRAINED', classifier=kind, selector='none',
                   boundary='NESTED_SAFE', permutation_id=0, feature_mode='EEG_timeseries', stage='three_pipeline')
    key = digest(dict(setting=setting, neural_key=data['neural_key'], channels=channel_names, plan=plan,
                      seed=config['seed'], inner_folds=config['inner_folds'], parameters=config['dl'] if kind=='scratch_cnn' else config['pretrained'],
                      device=str(neural_device(config)), torch=torch.__version__, engine=ENGINE_ID))
    directory = Path(cache_root)/'neural_runs'/key
    directory.mkdir(parents=True, exist_ok=True)
    save_json(directory/'setting.json', dict(setting=setting, channels=channel_names, pretrained=PRETRAINED_PROVENANCE if kind!='scratch_cnn' else None,
              shape=list(X.shape), plan_sha256=digest(plan), neural_manifest=data['neural_manifest'], result_id=key))
    save_json(directory/'plan.json', plan)
    n = len(data['y'])
    predictions, probabilities = np.full(n, -1), np.full((n, 4), np.nan)
    visits, folds, sessions = np.zeros(n, int), np.zeros(n, int), np.zeros(n, int)
    audits, fold_rows, history_rows = [], [], []
    E = embedding_key = None
    for fold in plan:
        train, test = np.asarray(fold['train']), np.asarray(fold['test'])
        token = f"ses-{fold['unit_session']:02d}_fold-{fold['fold']:02d}"
        checkpoint, audit_path = directory/(token+'.npz'), directory/(token+'.json')
        print(f"Subject {data['subject']:02d} | {len(indices)} channels | {kind} | session {fold['unit_session']} fold {fold['fold']}/{config['outer_folds']}", flush=True)
        if config['resume'] and checkpoint.exists() and audit_path.exists():
            with np.load(checkpoint) as z:
                pred, prob = z['pred'], z['probabilities']
            audit = json.loads(audit_path.read_text())
            if audit['train_rows'] != train.tolist() or audit['test_rows'] != test.tolist():
                raise ValueError('Neural checkpoint plan mismatch.')
            print('  Resumed completed fold.', flush=True)
        else:
            started = perf_counter()
            fold_seed = int(config['seed']+1009*data['subject']+37*fold['unit_session']+fold['fold'])
            if kind=='frozen_probe':
                if E is None:
                    E, embedding_key = frozen_embeddings(data, channel_names, encoder, config, cache_root)
                model = Pipeline([('scale', StandardScaler()), ('classifier', LogisticRegression(
                    C=config['pretrained']['probe_c'], max_iter=config['pretrained']['probe_max_iter'], solver='lbfgs'))])
                with threadpool_limits(limits=config['blas_threads']):
                    model.fit(E[train], data['y'][train])
                    prob, train_prob = model.predict_proba(E[test]), model.predict_proba(E[train])
                if model.classes_.tolist() != LABELS:
                    raise ValueError('Unexpected pretrained probe class order.')
                audit = dict(model_fit_rows=train.tolist(), scaler_fit_rows=train.tolist(), chosen_epoch=None,
                             inner_train_rows=[], inner_validation_rows=[], epoch_selection_rows=[], inner_scaler_fit_rows=[],
                             embedding_key=embedding_key, encoder_frozen=True, search_history=[], refit_history=[])
            else:
                model, input_values, audit = fit_neural_outer(X, data['y'], train, fold_seed, kind, config, encoder)
                prob = neural_probabilities(model, input_values[test], neural_device(config),
                                            config['dl']['batch_size'] if kind=='scratch_cnn' else config['pretrained']['batch_size'])
                train_prob = neural_probabilities(model, input_values[train], neural_device(config),
                                                  config['dl']['batch_size'] if kind=='scratch_cnn' else config['pretrained']['batch_size'])
                if config['save_neural_weights']:
                    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()}, directory/(token+'_weights.pt'))
                del model, input_values
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            pred = prob.argmax(axis=1)
            audit.update(train_rows=train.tolist(), test_rows=test.tolist(), fold_seed=fold_seed,
                         channels=channel_names, train_scores=metrics(data['y'][train], train_prob.argmax(axis=1)),
                         elapsed_seconds=perf_counter()-started,
                         fixed_normalization='microvolts /100 and zero padding' if kind!='scratch_cnn' else 'training-only channel mean/std')
            atomic_npz(checkpoint, pred=pred, probabilities=prob)
            save_json(audit_path, audit)
        if prob.shape != (len(test), 4) or len(pred)!=len(test) or not np.isfinite(prob).all() or not np.allclose(prob.sum(axis=1),1,atol=1e-5):
            raise ValueError('Invalid neural OOF predictions.')
        for k in ('model_fit_rows','scaler_fit_rows','inner_train_rows','inner_validation_rows','epoch_selection_rows','inner_scaler_fit_rows'):
            if not set(audit[k]).issubset(train) or set(audit[k]).intersection(test):
                raise ValueError(f'Leakage audit failed: {k}.')
        if set(audit['inner_train_rows']).intersection(audit['inner_validation_rows']):
            raise ValueError('Inner train/validation overlap.')
        predictions[test], probabilities[test] = pred, prob
        visits[test] += 1
        folds[test], sessions[test] = fold['fold'], fold['unit_session']
        scores = metrics(data['y'][test], pred)
        print(f"  Held-out BA={scores['balanced_accuracy']:.3f}, chosen epoch={audit['chosen_epoch']}, elapsed={audit['elapsed_seconds']/60:.2f} min.", flush=True)
        fold_rows.append(dict(**setting, fold=fold['fold'], unit_session=fold['unit_session'], chosen_epoch=audit['chosen_epoch'],
                              train_accuracy=audit['train_scores']['accuracy'], **scores))
        for phase, rows in (('inner_selection', audit['search_history']), ('outer_refit', audit['refit_history'])):
            history_rows.extend(dict(**r, fold=fold['fold'], unit_session=fold['unit_session'], phase=phase) for r in rows)
        audits.append(audit)
    if not np.all(visits==1) or not np.isfinite(probabilities).all():
        raise ValueError('Exactly one OOF prediction per original trial is required.')
    oof = data['metadata'].copy()
    oof['y_true'], oof['y_pred'], oof['outer_fold'], oof['unit_session'] = data['y'], predictions, folds, sessions
    for label in LABELS:
        oof[f'probability_{label}'] = probabilities[:, label]
    for name, value in setting.items():
        oof[name] = value
    row = dict(**setting, result_id=key, n_trials=n, OOF_predictions_per_trial=1,
               mean_train_accuracy=float(np.mean([a['train_scores']['accuracy'] for a in audits])),
               elapsed_seconds_sum=float(sum(a['elapsed_seconds'] for a in audits)), **metrics(data['y'], predictions))
    atomic_csv(oof, directory/'oof.csv')
    atomic_csv(pd.DataFrame(fold_rows), directory/'fold_scores.csv')
    if history_rows:
        atomic_csv(pd.DataFrame(history_rows), directory/'training_history.csv')
    atomic_csv(pd.DataFrame(confusion_matrix(data['y'],predictions,labels=LABELS), columns=[f'pred_{i}' for i in LABELS]).assign(true_class=LABELS), directory/'confusion.csv')
    save_json(directory/'result.json', row)
    result = dict(row=row, oof=oof, audits=audits, fold_scores=pd.DataFrame(fold_rows), directory=directory)
    active[key] = result
    return result

def run_neural_stage(datasets, channel_sets, config, cache_root, active, encoder, pipeline):
    kinds = ['scratch_cnn'] if pipeline=='DL' else config['pretrained']['modes']
    for data in datasets.values():
        for count in config['channel_counts']:
            for kind in kinds:
                evaluate_neural_task(data, channel_sets[count], kind, config, cache_root, active, encoder)
    return pd.DataFrame([r['row'] for r in active.values() if r['row']['pipeline']==pipeline])

def verify_paired_oof(ml_active, neural_active):
    by_subject = {}
    for result in list(ml_active.values())+list(neural_active.values()):
        if result['row'].get('permutation_id',0):
            continue
        oof = result['oof']
        columns = ['trial_id','y_true','outer_fold','unit_session']
        target = oof[columns].reset_index(drop=True)
        subject = result['row']['subject']
        if subject in by_subject:
            pd.testing.assert_frame_equal(target, by_subject[subject], check_dtype=False)
        else:
            by_subject[subject] = target
    print('Verified identical original trials, labels and outer folds across active pipelines/budgets.', flush=True)

def combined_reports(ml_active, neural_active, datasets, channel_sets, config, root):
    verify_paired_oof(ml_active, neural_active)
    all_results = list(ml_active.values())+list(neural_active.values())
    rows, sessions = [], []
    for result in all_results:
        row = dict(result['row'])
        row.setdefault('pipeline', 'ML')
        row['model'] = row['classifier']
        row['comparison'] = row['pipeline']+' | '+row['model']+' | '+row['selector']+' | '+row['boundary']
        rows.append(row)
        for session, group in result['oof'].groupby('session'):
            sessions.append(dict(subject=row['subject'], channel_count=row['channel_count'], pipeline=row['pipeline'],
                                 model=row['model'], selector=row['selector'], boundary=row['boundary'], scope=row['scope'],
                                 session=int(session), n_trials=len(group), **metrics(group.y_true,group.y_pred)))
    scores = pd.DataFrame(rows)
    if scores.empty:
        raise ValueError('No active completed results to report.')
    atomic_csv(scores, Path(root)/'all_subject_scores.csv')
    atomic_csv(pd.DataFrame(sessions), Path(root)/'session_oof_scores.csv')
    grouping = ['pipeline','model','selector','boundary','scope','channel_count','comparison']
    summary = scores.groupby(grouping,dropna=False).agg(
        n_subjects=('subject','nunique'), balanced_accuracy_mean=('balanced_accuracy','mean'),
        balanced_accuracy_sd=('balanced_accuracy','std'), accuracy_mean=('accuracy','mean'),
        macro_f1_mean=('macro_f1','mean'), train_accuracy_mean=('mean_train_accuracy','mean')).reset_index()
    changes = []
    for _, group in scores.groupby(['subject','comparison','scope']):
        reference = group[group.channel_count==128]
        if len(reference)!=1:
            raise ValueError('Each active comparison needs one 128-channel reference.')
        baseline = reference.iloc[0].balanced_accuracy
        for _, row in group.iterrows():
            changes.append(dict(subject=row.subject, comparison=row.comparison, pipeline=row.pipeline,
                                channel_count=row.channel_count, balanced_accuracy=row.balanced_accuracy,
                                reference_128_balanced_accuracy=baseline, change_from_128=row.balanced_accuracy-baseline))
    changes = pd.DataFrame(changes)
    gaps = []
    ml_scores = scores[scores.pipeline=='ML']
    for _, group in ml_scores.groupby(['subject','model','selector','scope','channel_count']):
        if set(group.boundary)=={'PAPER_ORDER','NESTED_SAFE'}:
            paper, safe = group.set_index('boundary').loc[['PAPER_ORDER','NESTED_SAFE'],'balanced_accuracy']
            first = group.iloc[0]
            gaps.append(dict(subject=first.subject, model=first.model, selector=first.selector, channel_count=first.channel_count,
                             paper_order_balanced_accuracy=paper,nested_safe_balanced_accuracy=safe,gap=paper-safe))
    expected=[]
    for subject in datasets:
        for count in channel_sets:
            if config['run_ml']:
                expected.extend(dict(subject=subject,channel_count=count,pipeline='ML',model=m,selector=s,boundary=b)
                                for m in config['classifiers'] for s in config['selectors'] for b in config['boundaries'])
            if config['run_dl']:
                expected.append(dict(subject=subject,channel_count=count,pipeline='DL',model='scratch_cnn',selector='none',boundary='NESTED_SAFE'))
            if config['run_pretrained']:
                expected.extend(dict(subject=subject,channel_count=count,pipeline='PRETRAINED',model=m,selector='none',boundary='NESTED_SAFE') for m in config['pretrained']['modes'])
    roster = pd.DataFrame(expected)
    keys = ['subject','channel_count','pipeline','model','selector','boundary']
    if scores.duplicated(keys).any():
        raise ValueError('Duplicate active subject evaluations.')
    completion = roster.merge(scores[keys+['result_id']],on=keys,how='left')
    completion['status'] = np.where(completion.result_id.notna(),'complete','pending_or_skipped')
    tables = dict(scores=scores,summary=summary,changes=changes,gaps=pd.DataFrame(gaps),completion=completion,sessions=pd.DataFrame(sessions))
    for name, table in tables.items():
        atomic_csv(table,Path(root)/(name+'.csv'))
    save_json(Path(root)/'completion.json',dict(expected=len(roster),complete=int(completion.status.eq('complete').sum()),
                                               partial=bool(completion.status.ne('complete').any())))
    return tables

def plot_three_pipeline_results(tables, ml_active, neural_active, config, root):
    summary = tables['summary']
    fig, ax = plt.subplots(figsize=(10,5))
    for name, frame in summary.groupby('comparison'):
        frame = frame.sort_values('channel_count')
        ax.errorbar(frame.channel_count,frame.balanced_accuracy_mean*100,
                    yerr=frame.balanced_accuracy_sd.fillna(0)*100,marker='o',capsize=3,label=name,
                    linestyle='--' if frame.boundary.iloc[0]=='PAPER_ORDER' else '-')
    ax.axhline(25,color='grey',linestyle=':',label='Four-class chance reference')
    ax.set(xlabel='Available EEG channels',ylabel='OOF balanced accuracy (%)',title='Subject means; error bars are between-subject SD')
    ax.set_xticks(sorted(config['channel_counts']))
    ax.legend(fontsize=8,bbox_to_anchor=(1.02,1),loc='upper left')
    fig.tight_layout()
    fig.savefig(Path(root)/'three_pipeline_channel_comparison.png',dpi=160,bbox_inches='tight')
    plt.show()
    results=list(ml_active.values())+list(neural_active.values())
    selected=[r for r in results if r['row']['subject']==config['subjects'][0] and r['row']['channel_count']==128]
    if selected:
        fig, axes=plt.subplots(1,len(selected),figsize=(4*len(selected),4),squeeze=False)
        for ax,result in zip(axes.flat,selected):
            cm=confusion_matrix(result['oof'].y_true,result['oof'].y_pred,labels=LABELS)
            ax.imshow(cm,cmap='Blues')
            for i in LABELS:
                for j in LABELS:
                    ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
            names=[config['class_names'][i] for i in LABELS]
            row=result['row']
            ax.set(xticks=LABELS,yticks=LABELS,xticklabels=names,yticklabels=names,xlabel='Predicted',ylabel='True',
                   title=f"{row['classifier']}\n{row['boundary']}")
        fig.tight_layout()
        fig.savefig(Path(root)/'first_subject_128_confusions.png',dpi=150,bbox_inches='tight')
        plt.show()
    histories=[]
    for result in neural_active.values():
        path=Path(result['directory'])/'training_history.csv'
        if path.exists() and result['row']['subject']==config['subjects'][0] and result['row']['channel_count']==128:
            frame=pd.read_csv(path)
            first=frame[['unit_session','fold']].drop_duplicates().iloc[0]
            frame=frame[(frame.unit_session==first.unit_session)&(frame.fold==first.fold)&(frame.phase=='inner_selection')]
            histories.append((result['row']['classifier'],frame))
    if histories:
        fig,axes=plt.subplots(1,len(histories),figsize=(6*len(histories),4),squeeze=False)
        for ax,(name,frame) in zip(axes.flat,histories):
            ax.plot(frame.epoch,frame.train_loss,label='Inner train')
            ax.plot(frame.epoch,frame.validation_loss,label='Inner validation')
            ax.set(title=name+' — first inner split',xlabel='Epoch',ylabel='Cross-entropy')
            ax.legend()
        fig.tight_layout()
        fig.savefig(Path(root)/'neural_inner_learning_curves.png',dpi=160,bbox_inches='tight')
        plt.show()

def export_three_pipeline_zip(ml_active, neural_active, config, root):
    root=Path(root)
    archive=root.parent/('three_pipeline_review_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.zip')
    readme='''Scores are proportions. PAPER_ORDER is an intentionally leaky diagnostic, not valid generalization. DL and pretrained evaluations use untouched outer test trials. Frozen embeddings are label-free and use verified external weights. Per-subject OOF predictions are the scoring units; outer folds are not independent subjects. This study subsets provider-processed 128-channel recordings in software. It is not an exact replication of the published feature-image DL implementation. Check completion.csv for pending/skipped requested evaluations. Raw EEG, embeddings and model-weight binaries are excluded. Source model checkpoint and architecture provenance are recorded.''' 
    with ZipFile(archive,'w',compression=ZIP_DEFLATED) as z:
        z.writestr('README.txt',readme)
        for path in sorted(root.rglob('*')):
            if path.is_file() and path.suffix in {'.csv','.json','.png','.md'}:
                z.write(path,'reports/'+str(path.relative_to(root)).replace('\\','/'))
        for pipeline,active in (('ML',ml_active),('neural',neural_active)):
            for key,result in active.items():
                directory=Path(result['directory'])
                for path in sorted(directory.iterdir()):
                    if path.is_file() and path.suffix in {'.csv','.json'}:
                        z.write(path,f'active_runs/{pipeline}/{key}/{path.name}')
    print(f'Review bundle: {archive} ({archive.stat().st_size/1e6:.2f} MB)',flush=True)
    return archive

ENGINE_ID = '0cdf49430e3ad49da85a93ebf9a092e8153894791bd0dc1cc66cb6452d952cdd'


## 6. Initialize and verify the pretrained checkpoint

Rerun from here after changing CONFIG. Checkpoint failures are reported immediately. Partial neural folds and completed ML folds can resume when all numerical settings match. Changing settings creates distinct cache identities. CPU works for implementation checks, but a GPU is recommended for the full study.


In [ ]:
CONFIG.update({'run_main': True, 'run_permutation_controls': False, 'n_permutations': 5, 'permutation_subjects': None, 'permutation_channel_counts': [128, 16], 'permutation_selectors': ['mrmr'], 'permutation_classifiers': ['svm'], 'plot_individual_subjects': True})
if CONFIG["run_permutation_controls"]:
    raise ValueError("This notebook reports real-label three-family comparisons. Keep run_permutation_controls=False.")
CONFIG["run_main"] = CONFIG["run_ml"]
RUN_CONFIG, RUN_ROOT, CACHE_ROOT = initialise_channel_study(CONFIG)
seed_torch(RUN_CONFIG["seed"], RUN_CONFIG)
ACTIVE_RESULTS = {}           # ML results only, preserving familiar variable names.
NEURAL_RESULTS = {}
STAGE_STATUS = []
PRETRAINED_ENCODER = load_frozen_encoder(RUN_CONFIG) if RUN_CONFIG["run_pretrained"] else None
save_json(RUN_ROOT / "pretrained_provenance.json", PRETRAINED_PROVENANCE)
save_json(RUN_ROOT / "runtime.json", dict(**environment_info(), torch=torch.__version__,
          cuda_available=torch.cuda.is_available(), device=str(neural_device(RUN_CONFIG)),
          evaluation_scope=RUN_CONFIG["evaluation_scope"], engine=ENGINE_ID))
print("Torch:", torch.__version__, "| Device:", neural_device(RUN_CONFIG))
print("Scope:", RUN_CONFIG["evaluation_scope"], "| outer folds:", RUN_CONFIG["outer_folds"])


## 7. Load identical trials for feature and neural inputs

`DATASETS[subject]` contains `F`, `y`, `metadata`, `schema`, `manifest`, plus `X` (original action-window EEG in volts) and `X_neural` (trial-local filtering, 200 Hz, microvolts). Raw EEG loading is checked against the exact trial-ID order used for features. The variable `F` remains the feature matrix; neural code uses `torch_F` for Torch functions.


In [ ]:
DATASETS = {}
for subject in RUN_CONFIG["subjects"]:
    data = prepare_subject_replication(subject, RUN_CONFIG)
    if RUN_CONFIG["run_dl"] or RUN_CONFIG["run_pretrained"]:
        data = attach_neural_input(data, RUN_CONFIG)
        save_json(RUN_ROOT / f"sub-{subject:02d}_neural_input.json", data["neural_manifest"])
    DATASETS[subject] = data
DATASET_SUMMARY = dataset_summary(DATASETS, RUN_ROOT)
display(DATASET_SUMMARY)
subject_data = DATASETS[RUN_CONFIG["subjects"][0]]
F, y, trial_metadata = subject_data["F"], subject_data["y"], subject_data["metadata"]
if "X" in subject_data:
    X, X_neural = subject_data["X"], subject_data["X_neural"]
print("Features:", F.shape, "| labels:", np.unique(y, return_counts=True))


## 8. Freeze spatial channel sets and shared outer folds

The geometry rule is the same canonical BioSemi128 farthest-point rule as our previous channel notebook. It uses no signals or labels and produces nested spatial subsets. Channel membership is shared across all subjects and model families. For within-session mode, make sure each class has enough trials in every requested session; the notebook raises an error instead of silently reducing folds.


In [ ]:
CHANNEL_SETS, CHANNEL_GEOMETRY = fixed_channel_sets(DATASETS, RUN_CONFIG, RUN_ROOT)
FEATURE_DIMENSIONS, ML_ROSTER, BASE_PLANS = preflight_channel_inputs(
    DATASETS, CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_CONFIG, RUN_ROOT)
display(FEATURE_DIMENSIONS)
if RUN_CONFIG["make_plots"]:
    plot_channel_sets(CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_ROOT, RUN_CONFIG["show_channel_names"])
WORKLOAD = []
for subject, data in DATASETS.items():
    folds = len(BASE_PLANS[subject])
    counts = len(RUN_CONFIG["channel_counts"])
    if RUN_CONFIG["run_ml"]:
        WORKLOAD.append(dict(subject=subject, pipeline="ML", outer_fits=folds*counts*len(RUN_CONFIG["classifiers"])*len(RUN_CONFIG["selectors"])*2,
                             extra="Optional inner SVM tuning adds fits."))
    if RUN_CONFIG["run_dl"]:
        WORKLOAD.append(dict(subject=subject, pipeline="DL", outer_fits=folds*counts,
                             extra="Each fold adds an inner epoch-selection fit, then an outer refit."))
    if RUN_CONFIG["run_pretrained"]:
        WORKLOAD.append(dict(subject=subject, pipeline="PRETRAINED", outer_fits=folds*counts*len(RUN_CONFIG["pretrained"]["modes"]),
                             extra="Frozen embeddings extracted once per subject/budget; finetuning adds inner fits."))
WORKLOAD = pd.DataFrame(WORKLOAD)
display(WORKLOAD)
atomic_csv(WORKLOAD, RUN_ROOT / "workload.csv")


## 9. Train ML: PAPER_ORDER and NESTED_SAFE

This section retains familiar MRMR/SVM settings, checkpointing and fitting-row audits. Selection is rerun after restricting the available channels. Safe and deliberately leaky branches use the same trial folds.


In [ ]:
if RUN_CONFIG["run_ml"]:
    ML_RESULTS = run_channel_stage(DATASETS, CHANNEL_SETS, CHANNEL_GEOMETRY, RUN_CONFIG,
                                  RUN_ROOT, CACHE_ROOT, ACTIVE_RESULTS, STAGE_STATUS, experiment="main")
    if ML_RESULTS.empty:
        print("No ML results completed; inspect stage_status.csv for skipped optional classifiers.")
    else:
        display(ML_RESULTS[["subject","channel_count","classifier","selector","boundary","balanced_accuracy","macro_f1"]])
else:
    print("ML disabled in CONFIG.")


## 10. Train the CNN from scratch

Two Conv1d blocks take all retained EEG channels together. Each model starts from random initialization. Epoch selection sees an inner validation subset of the outer training trials, then a fresh model is refitted on all outer training trials. The outer test data are used only after this refit. Progress prints every configured number of epochs.


In [ ]:
if RUN_CONFIG["run_dl"]:
    DL_RESULTS = run_neural_stage(DATASETS, CHANNEL_SETS, RUN_CONFIG, CACHE_ROOT,
                                  NEURAL_RESULTS, PRETRAINED_ENCODER, pipeline="DL")
    display(DL_RESULTS[["subject","channel_count","classifier","balanced_accuracy","macro_f1"]])
else:
    print("DL disabled in CONFIG.")


## 11. Train the pretrained transfer branch

Default `frozen_probe`: keep official CBraMod weights fixed; compute representations once for each exact channel budget, then fit standardization and logistic regression on outer training rows. There is no supervised full-data selection and no fine-tuning in this mode.

Optional `finetune`: train a small classifier and the encoder, with a lower encoder learning rate, inner epoch selection, and a fresh checkpoint reset for each outer refit. This is supervised transfer learning, not zero-shot inner-speech decoding. It is an extension to our experiment rather than a model evaluated in the Tasin paper.


In [ ]:
if RUN_CONFIG["run_pretrained"]:
    PRETRAINED_RESULTS = run_neural_stage(DATASETS, CHANNEL_SETS, RUN_CONFIG, CACHE_ROOT,
                                          NEURAL_RESULTS, PRETRAINED_ENCODER, pipeline="PRETRAINED")
    display(PRETRAINED_RESULTS[["subject","channel_count","classifier","balanced_accuracy","macro_f1"]])
else:
    print("Pretrained transfer disabled in CONFIG.")


## 12. Compare paired OOF results and audit completion

Scores are proportions (0.30 means 30%). Compare DL/pretrained primarily against ML NESTED_SAFE. Subject averages weight subjects equally; SD is computed across subjects, not across dependent CV folds. One-subject runs therefore have no between-subject SD.

`changes.csv` gives each subject/model's paired change from its own 128-channel result. `gaps.csv` records ML PAPER_ORDER minus NESTED_SAFE. The completion table lists every requested comparison, including missing optional classifiers. Do not claim a low-density model preserves useful decoding merely because both its score and the 128-channel score are near the 25% chance reference.


In [ ]:
THREE_TABLES = combined_reports(ACTIVE_RESULTS, NEURAL_RESULTS, DATASETS, CHANNEL_SETS, RUN_CONFIG, RUN_ROOT)
display(THREE_TABLES["summary"])
display(THREE_TABLES["changes"])
display(THREE_TABLES["gaps"])
display(THREE_TABLES["completion"])
print("Every active model has identical original-trial OOF folds within each subject.")


## 13. Channel-performance curves, confusion matrices and inner learning curves

Dashed PAPER_ORDER curves are the leaky diagnostic. Learning curves show the first inner split for the first subject at 128 channels; they are not outer-test learning curves. All underlying OOF predictions and per-fold training histories are saved for detailed analysis.


In [ ]:
if RUN_CONFIG["make_plots"]:
    plot_three_pipeline_results(THREE_TABLES, ACTIVE_RESULTS, NEURAL_RESULTS, RUN_CONFIG, RUN_ROOT)


## 14. Download the review bundle

Send this ZIP back for analysis. It contains configuration, model/weight provenance, fixed channel names, common split plans, subject/session scores, OOF predictions, per-fold fitting-row audits, selected ML features, epoch histories, confusion matrices and plots. It excludes raw EEG, embedding arrays and model-weight binaries. Exact caches stay in the working output directory for resume; download that directory separately if you need to resume across Kaggle sessions.


In [ ]:
REVIEW_ZIP = export_three_pipeline_zip(ACTIVE_RESULTS, NEURAL_RESULTS, RUN_CONFIG, RUN_ROOT)
display(FileLink(str(REVIEW_ZIP)))


## Suggested next experiments

1. Run the default one-subject comparison and inspect completion/OOF audits.
2. Keep the protocol fixed and use `subjects=None` to evaluate the full cohort. A cohort run is substantially longer.
3. Test `evaluation_scope="within_session"` as a separate experiment using the same scope for all families.
4. Compare frozen transfer against `pretrained.modes=["frozen_probe","finetune"]` when GPU time permits.
5. Only after the safe baseline is understood, predeclare additional architecture/hyperparameter searches inside training folds and stronger randomized-label controls. Repeated outer-score inspection is exploratory model development, not an independent confirmatory test.

The paper's feature-image ResNet18/VGG16 branch is not implemented here because its rendering, selection mapping and pretrained-weight settings are insufficiently specified. The pretrained EEG path above avoids inventing those details.
